In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import scipy.stats as stats
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from itertools import combinations
import scipy.optimize as optimize
from scipy.stats.contingency import association
import plotly.graph_objects as go
pd.set_option("display.max_columns",None)
import statsmodels.api as sm
import seaborn as sns
from scipy.linalg import fractional_matrix_power

from matplotlib.ticker import NullFormatter

import ipywidgets as widgets
from rich.console import Console
from rich.text import Text
from rich.table import Table
import pyxirr
from pandas.tseries.offsets import DateOffset
from itertools import takewhile
console = Console()
pd.options.display.float_format = '{:,.2f}'.format

import risk_kit as rk
import os
import cvxpy as cp
import importlib
from concurrent.futures import ProcessPoolExecutor
import time
from itertools import product
import kaleido
import gc

In [ ]:
"""
GLOBAL VARIABLES DECLARADATION : 
Do not change the variable names. The values within it can be changed. 
"""
scen = ["Base", "Optimistic", "Pessimistic"]

tail_prob = [0.01, 0.99]  # Adjust the values based on the number of scenarios apart from the "Baseline" scenario
#           [Optimistic, Pessimistic]
# ^ Make sure the values are aligned with the "scen" list.

rating_order = ["AAA","AA","A","BBB","BB","B","CCC","D"]
buckets = ['Current', '30-59 DPD', '60-89 DPD', '90+ DPD']
start_reporting_date = pd.to_datetime("2023-03-31")
end_reporting_date = pd.to_datetime("2025-12-31")
#sanction_rating_order = ["AAA","AA","A","BBB","BB","B","CCC"]
rating_order = ["AAA","AA","A","BBB","BB","B","CCC","D"]
ScenarioColorCode = ["navy","green","red"]

CCF = {"Commercial" : 45, # 45%
       "Retail" : 75,     # 75%
       "Corporate" : 50,  # 50%
       "SME" : 75}        # 75%

rho_CCF = 0.12 # 12%

LGD = {"Commercial": 65, #65%
      "Corporate" : 70,  #65%
      "Retail" : 60,     #60%
      "SME" : 60}        #60%

rho_LGD = 0.12 # 12%

rho = 0.12 # Correlation to be used in the vasicek model for conversion of TTC- TM to PIT-TM for Ratings and DPD

reporting_dates_list = pd.date_range(start= start_reporting_date, end=end_reporting_date,freq="QE")
reporting_dates_list = reporting_dates_list.strftime("%Y-%m-%d")

Z_scenarios = ["Z_"+s for s in scen]

In [ ]:
data = pd.read_excel("Sample_Portfolio.xlsm", sheet_name= 'Portfolio')

In [ ]:
console.print(f"Raw Data dimension : {data.shape}", style="bold")
data.head()

In [ ]:
# Making sure the dates are in a specific format:

data["Maturity_Date"] = pd.to_datetime(data["Maturity_Date"], format ="%d-%m-%Y")
data["Sanction_Date"] = pd.to_datetime(data["Sanction_Date"], format ="%d-%m-%Y")
data["Reporting_Date"] = pd.to_datetime(data["Reporting_Date"], format ="%d-%m-%Y")

In [ ]:
# Origination Rating distribution plot :

raw_rating_dist = pd.DataFrame(data["Origination_Rating"].value_counts().reindex(rating_order[:-1], fill_value=0))
raw_rating_dist = raw_rating_dist.rename(columns={"count":"Before_count"})

before_loan_type_count = pd.DataFrame(data.groupby(data["Type_of_Loan"])["Customer_ID"].count()).rename(columns={"Customer_ID":"Before_count"})

before_segment_type_count = pd.DataFrame(data.groupby(data["Segment"])["Customer_ID"].count()).rename(columns={"Customer_ID":"Before_count"})

before_total = raw_rating_dist["Before_count"].sum(axis=0)

# Filtering the Data to only consider those loans that are active on the Sanction Date : 

filtered_data = data[(data["Sanction_Date"]<= "2023-03-31")]
filtered_data = filtered_data[(filtered_data["Maturity_Date"]>"2023-03-31") | (filtered_data["Type_of_Loan"] == "Revolving")]

reporting_date = pd.to_datetime("2023-03-31")

after_filter_rating_count = pd.DataFrame(filtered_data["Origination_Rating"].value_counts().reindex(rating_order[:-1], fill_value=0)).rename(columns={"count":"After_count"}) # Excluding the "D" rating

combned_rating_data = pd.concat([after_filter_rating_count, raw_rating_dist], axis=1)

after_filter_loan_type_count = pd.DataFrame(filtered_data["Type_of_Loan"].value_counts()).rename(columns={"count":"After_count"})

combined_loan_type_data = pd.concat([after_filter_loan_type_count,before_loan_type_count],axis=1)

after_filter_segment_count = pd.DataFrame(filtered_data["Segment"].value_counts()).rename(columns={"count":"After_count"})
combined_segment_data = pd.concat([before_segment_type_count,after_filter_segment_count],axis=1)

width = 0.35

x_rating = np.arange(len(rating_order[:-1]))

x_loantype = np.arange(len(before_loan_type_count.index))

x_segmenttype = np.arange(len(combined_segment_data.index))

fig, ax = plt.subplots(nrows=2,ncols=2, figsize=(10,6))

console.print(f" Total number of borrowers before filtering: {before_total}\n",style="Bold")
console.print(f" Total number of borrowers after filtering: {after_filter_rating_count['After_count'].sum()}\n",style="Bold")

# Rating Distribution Plot : 

ax[0,0].bar(x_rating + width /2,combned_rating_data["After_count"],width,color="#4B716C", label="After Filtering") # After filtering
ax[0,0].bar(x_rating - width /2,combned_rating_data["Before_count"],width,color="#36454F", label="Before Filtering") # Before filtering
ax[0,0].set_xlabel("Ratings")
ax[0,0].set_ylabel("Count of loans")
ax[0,0].set_title("Number of borrowers at each rating grade")
ax[0,0].set_xticks(x_rating)
ax[0,0].set_xticklabels(rating_order[:-1])
ax[0,0].legend()

# Loan type plot :
ax[0,1].bar(x_loantype + width/2, combined_loan_type_data["After_count"], width, color= "#9AA899",label="After Filtering") # After Filtering
ax[0,1].bar(x_loantype - width/2, combined_loan_type_data["Before_count"], width, color="#4A5D4E", label="Before Filtering") # Before filtering
ax[0,1].set_xlabel("Type of Loan")
ax[0,1].set_ylabel("Count of loans")
ax[0,1].set_title("Number of loans w.r.t loan type")
ax[0,1].set_xticks(x_loantype)
ax[0,1].set_xticklabels(combined_loan_type_data.index)
ax[0,1].legend()


ax[1,0].bar(x_segmenttype + width/2, combined_segment_data["After_count"], width, color= "#69845E",label="After Filtering") # After Filtering
ax[1,0].bar(x_segmenttype - width/2, combined_segment_data["Before_count"],width, color="#35562C",label="Before Filtering") # Before Filtering
ax[1,0].set_xlabel("Segment Type")
ax[1,0].set_ylabel("Count of loans")
ax[1,0].set_title("Number of loans w.r.t Segment")
ax[1,0].set_xticks(x_segmenttype)
ax[1,0].set_xticklabels(combined_segment_data.index)
ax[1,0].legend()

ax[1,1].set_visible(False)
plt.tight_layout()
flat_axes = ax.flatten()
for axes in flat_axes :
    axes.spines["right"].set_visible(False)
    axes.spines["top"].set_visible(False)
plt.show()

# Deleting temporary DataFrames created in this block : 
del before_loan_type_count, before_segment_type_count, before_total , after_filter_rating_count, combned_rating_data, after_filter_loan_type_count, combined_loan_type_data
del after_filter_segment_count, combined_segment_data
gc.collect()

In [ ]:
# Creating a copy of the original dataset and forecasting the whole dataset for the next 2 years on a quarterly basis, from Q1 2023 - Q4 2025.

forecasting_data = filtered_data.copy()

forecasting_data["Sanction_Date"] = pd.to_datetime(forecasting_data["Sanction_Date"], dayfirst=True)
forecasting_data["Maturity_Date"] = pd.to_datetime(forecasting_data["Maturity_Date"], dayfirst=True) # dayfirst = True makes sure the format is DD/MM/YYY
forecasting_data["Reporting_Date"] = pd.to_datetime(forecasting_data["Reporting_Date"], dayfirst=True)

forecasting_data["end_date_amort"] = forecasting_data["Maturity_Date"].clip(upper=end_reporting_date)
forecasting_data["start_date_clipped"] = forecasting_data["Sanction_Date"].clip(lower=start_reporting_date)

is_amort_bullet = forecasting_data["Type_of_Loan"].isin(["Amortising", "Bullet"])
forecasting_data["end_date"] = forecasting_data["end_date_amort"].where(
    is_amort_bullet, 
    forecasting_data["Reporting_Date"].clip(upper=end_reporting_date))

# Identify the theoretical quarter-end for the current end date :

curr_q_end = forecasting_data["end_date"].dt.to_period("Q").dt.to_timestamp("Q").dt.normalize()
prev_q_end = (forecasting_data["end_date"].dt.to_period("Q") - 1).dt.to_timestamp("Q").dt.normalize()

# If end_date didn't survive until the actual quarter end, drop it back to the previous quarter end
forecasting_data["reporting_end_date"] = np.where(forecasting_data["end_date"] == curr_q_end,
                                                  curr_q_end,
                                                  prev_q_end)

forecasting_data["Quarter_Count"] = (
    (forecasting_data["reporting_end_date"].dt.year - forecasting_data["start_date_clipped"].dt.year) * 4 +
    (forecasting_data["reporting_end_date"].dt.quarter - forecasting_data["start_date_clipped"].dt.quarter))

# Clip to 0 for historical loans that matured prior to March 31, 2023
forecasting_data["Total_Periods"] = (forecasting_data["Quarter_Count"] + 1).clip(lower=0)

# 6. Replicate Rows
forecasting_data = forecasting_data.loc[forecasting_data.index.repeat(forecasting_data["Total_Periods"])].reset_index(drop=True)

# 7. Map final clean timeline reporting dates
if not forecasting_data.empty:
    forecasting_data["Quarter_Index"] = forecasting_data.groupby("Customer_ID").cumcount()
    forecasting_data["Reporting_Date"] = (
        forecasting_data["start_date_clipped"].dt.to_period("Q") + forecasting_data["Quarter_Index"]).dt.to_timestamp("Q").dt.normalize()

#The Following counts the number of quarters elapsed between the Sanction Date and the first reporting date of that loan, this will be used later : 

forecasting_data["Q_Count_for_Rating_Est."] = (
    (forecasting_data["Reporting_Date"].dt.year - forecasting_data["Sanction_Date"].dt.year) * 4 +
    (forecasting_data["Reporting_Date"].dt.quarter - forecasting_data["Sanction_Date"].dt.quarter))

console.print(f" Data Dimension : {forecasting_data.shape}",style="bold")
console.print(f" Unique borrowers : {forecasting_data['Customer_ID'].nunique()}",style="bold")

forecasting_data

# Macroeconomic Modelling

In [ ]:
historical_mevs = pd.read_excel("Sample_Portfolio.xlsm",sheet_name = "Macroeconomic variable")

# Adjusting the dates : 
historical_mevs["observation_date"] = historical_mevs["observation_date"] + pd.offsets.QuarterEnd(0)

pit_score = stats.norm.ppf(historical_mevs["Default Rates"])
rho = pit_score.var()/(1+pit_score.var())
TTC_avg_score = pit_score.mean()
TTC_threshold = TTC_avg_score * np.sqrt(1-rho)
historical_mevs["Z"] = (TTC_threshold - stats.norm.ppf(historical_mevs["Default Rates"])*np.sqrt(1-rho))/np.sqrt(rho)

# Historical_Z

# Plot : 
fig,ax = plt.subplots(figsize=(11,5))# nrows=1,ncols=2,figsize=(12,6))
ax.plot(historical_mevs["observation_date"],historical_mevs["Default Rates"], label="default rates",color="#36454F")
ax.set_ylabel("Default Rates",fontsize=12)
ax.set_xlabel("Time period",fontsize=12)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax2 = ax.twinx()

ax2.plot(historical_mevs["observation_date"],historical_mevs["Z"], label="Extracted Z",color="#040348", linestyle="--")
ax2.set_ylabel("Extracted Macroeconomic index (Z)",fontsize=12)
fig.legend()
plt.title("Historical Default Rates v/s Extracted Macroeconomic Index", loc="left", fontsize=15,pad = 15,fontweight="bold", style="italic", color="darkslategrey")
ax.grid(axis="x", alpha = 0.3)
plt.savefig("8_Default rates vs Extracted Z.png", bbox_inches="tight")
ax.spines["left"].set_visible(False)
plt.show()
display(historical_mevs)
mev_data = historical_mevs.copy()

In [ ]:
def check_stationarity(timeseries):
    result = adfuller(timeseries.values.flatten())
    print('ADF Statistic: %f' % result[0])
    print(f"p-value: {result[1]:.5f}\n")

In [ ]:
cols_acf_pacf = historical_mevs.columns.to_list()[2:]
n_rows = len(cols_acf_pacf)

fig, ax = plt.subplots(n_rows,2, figsize=(12,10))
flat_ax = ax.flatten()

for idx, col in enumerate(cols_acf_pacf):  
    acf_ax = flat_ax[idx*2]
    pacf_ax = flat_ax[idx*2+1]
    
    plot_acf(historical_mevs[col], lags=10, ax = acf_ax)
    acf_ax.set_title(f"ACF of {col}")
    acf_ax.set_xlabel("Lags")
    plot_pacf(historical_mevs[col], lags = 10, ax = pacf_ax)
    pacf_ax.set_title(f"PACF of {col}")
    pacf_ax.set_xlabel("Lags")
plt.tight_layout()

In [ ]:
print("----------------STATIONARITY CHECK ---------------- \n")

for col in cols_acf_pacf:

    print(f"Variable : {col}--------------------")
    check_stationarity(historical_mevs[col])

    print(f"After taking the first difference")
    check_stationarity(historical_mevs[col].diff().dropna())
    
    print(f"After taking the Second difference")
    check_stationarity(historical_mevs[col].diff().diff().dropna())
    
    print(f"After the Yeo Johnson Transformation")
    x_transformed, max_log_lambda = stats.yeojohnson(historical_mevs[col])
    x_transformed = pd.Series(x_transformed)
    check_stationarity(x_transformed)

In [ ]:
# ————————————————————————————
# Raw Macroeconomic Data Plot : 
# ————————————————————————————

mev_list = historical_mevs.columns.to_list()[2:-1]

plt_data = historical_mevs[["observation_date", "UNRATE","HPI","GDP"]]
plt_data = plt_data.set_index("observation_date")
fig, ax = plt.subplots(nrows= len(mev_list),ncols =2,figsize=(12,7))
flat_ax = ax.flatten()
mev_colorList = ["darkblue", "orange", "orangered"]
for idx, axes in enumerate(flat_ax):
    if idx < len(mev_list):
        axes.plot(plt_data[mev_list[idx]],label=mev_list[idx], color = mev_colorList[idx])
        axes.set_ylabel(mev_list[idx])
        axes.set_title(mev_list[idx])
        axes.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    else:
        axes.set_visible(False)
plt.tight_layout()

In [ ]:
historical_mevs_1 = historical_mevs
historical_mevs_1 = historical_mevs_1.set_index("observation_date")
historical_mevs_1["UNRATE"] = historical_mevs_1["UNRATE"].diff()
historical_mevs_1["HPI"] = historical_mevs_1["HPI"].diff()
historical_mevs_1["GDP"] = historical_mevs_1["GDP"].diff()
historical_mevs_1["Z"] = historical_mevs_1["Z"].diff()
historical_mevs_1 = historical_mevs_1.dropna()

cols_to_plot = historical_mevs_1.columns[1:]
data_after_first_transformation = historical_mevs_1.iloc[:,1:]

fig, ax = plt.subplots(nrows = len(mev_list),ncols = 2, figsize=(12,6))
flat_ax = ax.flatten()

for idx, axes in enumerate(flat_ax):
    if idx < len(mev_list):
        axes.plot(data_after_first_transformation[mev_list[idx]], label = f"{mev_list[idx]}", color="orange",alpha=0.7)
        axes.set_title(f"{mev_list[idx]}")
        axes.set_ylabel(f"First Difference :{mev_list[idx]}")
        axes.set_xlabel(f"Time Period")
    else:
        axes.set_visible(False)
plt.tight_layout()


cols_acf_pacf = historical_mevs_1.columns.to_list()[2:]
n_rows = len(cols_acf_pacf)

fig, ax = plt.subplots(n_rows,2, figsize=(12,10))
flat_ax = ax.flatten()

for idx, col in enumerate(cols_acf_pacf):  
    acf_ax = flat_ax[idx*2]
    pacf_ax = flat_ax[idx*2+1]
    
    plot_acf(historical_mevs_1[col], lags=10, ax = acf_ax)
    acf_ax.set_title(f"ACF of Transformed (First difference) {col}")
    acf_ax.set_xlabel("Lags")
    plot_pacf(historical_mevs_1[col], lags = 10, ax = pacf_ax)
    pacf_ax.set_title(f"PACF of Transformed (First difference) {col}")
    pacf_ax.set_xlabel("Lags")
plt.tight_layout()

### Johansen Test

In [ ]:
endo = historical_mevs.iloc[:,2:]

result_with_DefRates = coint_johansen(endo,
                                      det_order = 0,
                                      k_ar_diff = 1)

print(f"Trace Statistic : {result_with_DefRates.trace_stat}\n"
      f"Critical Values (95%) : {result_with_DefRates.trace_stat_crit_vals[:,1]}\n")

"""
Checking if Cointegrating Equations exists. 
Decision : coint_rank_val must be >= 1 to proceed with the VECM Model.

"""
coint_rank_val = 0
for i in range(len(result_with_DefRates.trace_stat)):
    if result_with_DefRates.trace_stat[i] > result_with_DefRates.trace_stat_crit_vals[:,1][i]:
        coint_rank_val = i+1
    else:
        break
print(f" Variables in the model : {endo.columns}\n"
      f"=========================\n Cointegrating Rank : {coint_rank_val}\n=========================\n")

model_1 = VECM(endo, k_ar_diff=1, coint_rank=coint_rank_val)#, deterministic="n")
results = model_1.fit()
print(f"VECM Model on variables : {endo.columns}")
print(results.summary())
steps = len(reporting_dates_list)
Forecast = results.predict(steps=steps)
#pd.DataFrame(Forecast).to_excel("Forecast1.xlsx")
#pd.DataFrame(results.fittedvalues).to_excel("Fitted1.xlsx")

In [ ]:
# ————————————————————————————————————————
# Forecasting of Macroeconomic variables : 
# ————————————————————————————————————————

forecast_point, lower_ci, upper_ci = results.predict(steps=12, alpha= 0.05) # 5%

var_names = results.names

arrays = [forecast_point, lower_ci, upper_ci]

mev_forecast_data = {}
# for i, var in enumerate(var_names):
#     # Create columns grouped by each variable
#     mev_forecast_data[f"{var}_Base"] = forecast_point[:, i]
#     mev_forecast_data[f"{var}_Optimistic"] = lower_ci[:, i]
#     mev_forecast_data[f"{var}_Pessimistic"] = upper_ci[:, i]

for i, var in enumerate(var_names):
    for s, data_array in zip(scen, arrays):
        mev_forecast_data[f"{var}_{s}"] = data_array[:, i]
    
# 4. Convert the dictionary into a beautiful Pandas DataFrame
mev_forecast = pd.DataFrame(mev_forecast_data).set_index(reporting_dates_list).reset_index()
mev_forecast= mev_forecast.rename(columns={"index":"observation_date"})
mev_forecast["observation_date"] = pd.to_datetime(mev_forecast["observation_date"])

# ————————————————————————
# ADJUSTMENTS : 
# ————————————————————————

"""
Note : Lower Percentile for GDP showcases a bad economy, thus the values needs to be adjusted. Similarly the Column names needs to be adjusted for Z 
as a lower value of Z indicates a bad economy and vice - versa.

"""
mev_forecast = mev_forecast.rename(columns=({"GDP_Optimistic":"GDP_Pessimistic","GDP_Pessimistic":"GDP_Optimistic",
                                             "HPI_Optimistic":"HPI_Pessimistic","HPI_Pessimistic":"HPI_Optimistic",
                                            "Z_Optimistic":"Z_Pessimistic","Z_Pessimistic":"Z_Optimistic"}))
n_rows = round(forecast_point.shape[1]/2)

#Plotting ——————————————————————
fig, ax = plt.subplots(n_rows,2, figsize=(19,10))
flat_ax = ax.flatten()

for idx, axes in enumerate(flat_ax):
    if idx < len(var_names):
        axes.plot(mev_forecast["observation_date"],mev_forecast.filter(regex=f"{var_names[idx]}_{scen[0]}"),color="Blue", label=f"{var_names[idx]}_{scen[0]}")
        axes.plot(mev_forecast["observation_date"],mev_forecast.filter(regex=f"{var_names[idx]}_{scen[1]}"),color="Green",linestyle="--", label=f"{var_names[idx]}_{scen[1]}")
        axes.plot(mev_forecast["observation_date"],mev_forecast.filter(regex=f"{var_names[idx]}_{scen[2]}"),color="Red",linestyle="--",label=f"{var_names[idx]}_{scen[2]}") 
        axes.set_title(f"{var_names[idx]}_Forecast")
        
        if var_names[idx]!="Z":
            axes.yaxis.set_major_formatter(PercentFormatter(xmax=1))
        
        axes.tick_params(axis="x",rotation=20)
        axes.grid(axis="y",alpha=0.4)
        axes.legend(loc="upper left")
        axes.set_xlabel("Time Period (Quarterly)")
    else:
        axes.set_visible(False)
fig.tight_layout()
plt.show()
display(mev_forecast)

In [ ]:
nrows = len(mev_list)
fig, ax = plt.subplots(nrows,2, figsize=(12,6))
flatten_ax = ax.flatten()
ScenarioColorCode = ["navy","green","red"]
for idx, axes in enumerate(flatten_ax):
    if idx < len(mev_list):
        axes.plot(historical_mevs["observation_date"], historical_mevs[mev_list[idx]], label=f"{mev_list[idx]}", color= "darkslategrey")
        axes.axvline(historical_mevs["observation_date"].iloc[-1], linestyle="--", color="black", alpha=0.5)
        for i, s in enumerate(scen):
            axes.plot(mev_forecast["observation_date"], mev_forecast[f"{mev_list[idx]}_{s}"], color= ScenarioColorCode[i], linestyle="--")
        axes.set_title(f"{mev_list[idx]}")
        axes.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    else:
        axes.set_visible(False)
#fig.legend(bbox_to_anchor=(1,1))
fig.suptitle("Macroeconomic variable and scenario Projections", fontsize=15, style="italic", color="black", y= 1.03)
plt.tight_layout()

In [ ]:
# ————————————————————————
#  AR(P) AND VAR MODELS : 
# ————————————————————————

# from itertools import combinations
# from statsmodels.tsa.api import VAR
# combination = combinations(data_mev.columns[1:-1],r=2)

# file_path = "Macroeconomic Variables Regression - AR and VAR Model_1.md"
# with open(file_path, "a") as f:
#     f.write("AR and VAR Regression Model Results - on Raw Data\n\n")

# combination_list = []
# data_columns = data_mev.columns[1:-1]
# for c in range(1,4):
#     combination = list(combinations(data_columns,r=c))
#     combination_list.extend(combination)

# lag = 1
# for cols in combination_list:
#     data_f = data_mev[list(cols)].copy()
#     data_VAR = data_mev[list(cols)].copy()
    
#     if len(cols)==1 : # Autoregressive model : 
        
#         for lag in range(1,6): # The number of lags (from 1 to 5)
            
#             data_f[f"{list(cols)[0]}_lag_{lag}"] = data_f[list(cols)[0]].shift(lag) # 1 period lag
#             data_f = data_f.dropna()
#             #X_col = data_f[f"{list(cols)[0]}_lag"]
#             X_col = data_f.filter(regex=f"{list(cols)[0]}_lag_")
#             X = sm.add_constant(X_col)
#             Y = data_f[list(cols)]
            
#             model = sm.OLS(Y,X).fit()
#             print(f"One Variable AR Model with {lag} lag for Macro variable : {cols[0]}\n")
#             display(model.summary())
#             print("\n=======================================================******************************************=================================================\n")

#             with open(file_path, "a") as f:
#                 f.write(f"One Variable AR Model with {lag} lag for Macro variable : {cols[0]}\n")
#                 f.write(model.summary().as_text())
#                 f.write("\n=======================================================******************************************=================================================\n")
    
#     else: # Vector Autoregressive model : 
#         for lag in range(1,6):            
#             model_VAR = VAR(mev_data[list(cols)])
        
#             results = model_VAR.fit(maxlags=lag)
#             print(f"VAR Model for Macroeconomic Variable : {cols} \n")
#             display(results.summary())
#             print("\n=======================================================******************************************=================================================\n")
#             result = str(results.summary())
#             print(f" Is the VAR Model Stationary? {results.is_stable(verbose=True)}")
#             with open(file_path, "a") as f:
#                     f.write(f"VAR Model for Macroeconomic Variable : {cols} \n")
#                     f.write(result)
#                     f.write("\n=======================================================******************************************=================================================\n")

## Probability of Default Estimation : 

In [ ]:

rating_count = filtered_data.groupby("Origination_Rating")["Customer_ID"].count().reset_index()

rating_count['Origination_Rating'] = pd.Categorical(rating_count['Origination_Rating'], categories= rating_order[:-1], ordered=True) # Excluding the "D" rating as they are hardcoded as 100%.
rating_count = rating_count.sort_values("Origination_Rating").reset_index(drop=True)
rating_count = rating_count.rename(columns={"Customer_ID":"Number of borrowers"})
# Creating the Dataframe for PD computation (using Pluto Tasche method): 
total_borrowers = rating_count["Number of borrowers"].sum()
rating_count["Number of borrowers at or below current rating"] = rating_count["Number of borrowers"][::-1].cumsum()[::-1]

# The PD estimation is done under the assumption that there are 0 defaulters under each rating grade.
rating_count["Defaulters"]=0

rating_count["PD estimation_Uncorrelated (%)"] = (1-(1-0.75)**(1/rating_count["Number of borrowers at or below current rating"]))*100

asset_correlation = 0.12

trial_runs = 100000
target_cdf = 0.25

np.random.seed(42)
Z = np.random.normal(0,1,trial_runs)

def pluto_tasche_correlated(p_guess, k, n, Z, rho, target):
    p_guess = np.clip(p_guess, 1e-7,1-1e-7)
    
    p_star = stats.norm.cdf((stats.norm.ppf(p_guess)-np.sqrt(asset_correlation)*Z)/np.sqrt(1-asset_correlation))
    p_star = np.clip(p_star,1e-15,1-1e-15)
    
    #Computing the Cumulative binomial probabilites across all trial runs
    cumulative_prob = stats.binom.cdf(k=k,n=n, p=p_star)
    
    # Computing the expected value (mean) of the resulting probabilities (p*)
    average = np.mean(cumulative_prob)
    
    # Objective : The difference should be close to or exactly 0 ( ie., the mean should be 25%)
    return average - target

# Running the optimisation across all the grades

correlated_pd = []

for i in range(len(rating_count)):
    k = 0
    n = rating_count["Number of borrowers at or below current rating"].iloc[i]
    optimised_p = optimize.brentq(pluto_tasche_correlated,
                                  a = 1e-5, b= 1-1e-5,
                                  args = (k,n,Z,asset_correlation,target_cdf))
    
    correlated_pd.append(optimised_p*100)

rating_count["PD_estimation_correlated (%)"] = correlated_pd
                     
rating_count["Lambda"] = -np.log(1-(rating_count["PD_estimation_correlated (%)"]/100))

# —————————————
# Plotting : 
# —————————————

fig, ax = plt.subplots(figsize=(14,6))

ax.plot(rating_count["Origination_Rating"],rating_count["PD_estimation_correlated (%)"], label="PD with correlated borrowers",marker="p", color="midnightblue")
ax.plot(rating_count["Origination_Rating"],rating_count["PD estimation_Uncorrelated (%)"], label = "PD asssuming independent borrowers",marker="o", linestyle="--", color="grey")
ax.set_xlabel("Origination Credit Rating", color='#010B13', fontsize=12)
ax.set_ylabel("Probability of default", color='#010B13',fontsize=12)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
plt.legend()
ax.grid(alpha=0.25,color="grey",linestyle="-",axis="y")
ax.set_title("Pluto Tasche : Probability of default estimation", loc = "left",pad=15,fontsize=15,fontweight="bold", style="italic", color="darkslategrey")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
#plt.savefig("PD_Computation_PLutoTasche",transparent=True, dpi=400)

plt.show()

# Table : 
pd_table = Table()
pd_table = Table(title="Probability of Default", show_header=True, header_style="bold #36454F")
pd_table.add_column("Origination Rating", width= 14, justify="center",vertical="middle", style="bold")
pd_table.add_column(f"PD estimate of Uncorrelated borrowers\n(in %)", width= 14,justify="center")
pd_table.add_column(f"PD estimate of Correlated borrowers\n(in %)", width= 14,justify="center")
for i in range(len(rating_count)):
    pd_table.add_row(f"{rating_count["Origination_Rating"].iloc[i]}",
                    f"{rating_count["PD estimation_Uncorrelated (%)"].iloc[i]:.4f}",
                    f"{rating_count["PD_estimation_correlated (%)"].iloc[i]:.4f}",end_section=True)
console.print(pd_table)
console.print(" Note : Lambda is computed on the 'PD_estimation_correlated (%)' Column",style="bold")

In [ ]:
def hazard_rate(t = 1000):

    lambda_map = rating_count.set_index("Origination_Rating")["Lambda"]
    rating_lambda_map = dict(zip(rating_order,lambda_map))
    pd_curve = pd.DataFrame(np.arange(0,t,1),columns=["Year"])
    
    for r in rating_order :
        pd_curve[r] = (1- np.exp(-(rating_lambda_map[r]*pd_curve["Year"])) if r!="D" else 1)
    pd_curve.iloc[0,-1] = 0.00 # Setting the "D" rating month 0 value as 0.00
    
    #Plotting
    fig, ax = plt.subplots(figsize=(12,6))
    for r in rating_order:
        if r=="D":
            ax.plot(pd_curve["Year"], pd_curve[r], label = r, linewidth=2,color="Black")
        else:
            plt.plot(pd_curve["Year"], pd_curve[r], label = r)
    ax.set_ylabel("Probability of Default", fontweight="semibold", color="#36454F")
    ax.set_xlabel("Time Period (Quarters)", fontweight="semibold",color="#36454F")
    ax.legend(bbox_to_anchor=(1.1,1))
    ax.set_title("Cumulative PD Structure - The Hazard Rate Model",fontsize=14, loc="left", fontweight="bold", style="italic", color="darkslategrey")
    ax.grid(alpha=0.5,axis="y",color="grey",linestyle="-")
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    #plt.savefig("Hazard_Rate.png",dpi=600)
widgets.interact(hazard_rate,t=widgets.IntSlider(value=50,min=5,max=100, step=4,description="Time Period", continuous_update=True))

### We have the Macroeconomic index for Base, Pessimistic and Optimistic Scenario in our 'mev_forecast' dataset, and our estimate of the PD value in 'rating_count' dataset we now need to map those Z values with our forecasting_data and assign the PD to each borrower. 

In [ ]:
"""
Getting the PMT, RAW O/s balance for bullet and amrortising loans. 
For bullet loans, a simple interest is assumed and the borrower pays interest amount on a monthly basis. 
   
"""

is_amortising_loan = forecasting_data["Type_of_Loan"].isin(["Amortising"]) 

rate = forecasting_data["Origination_Interest_Rate"]/12  # Monthly interest rate
principal_amount =forecasting_data["Sanction_Amount (in USD)"]
periods = forecasting_data["Tenure"]

emi = (principal_amount * rate * ((1+rate)**periods))/((1+rate)**periods-1)

forecasting_data["PMT"] = np.where(is_amortising_loan== True,
                                            emi,0)

forecasting_data["MOB"] = ((forecasting_data["Reporting_Date"].dt.year - forecasting_data["Sanction_Date"].dt.year) * 12) + (forecasting_data["Reporting_Date"].dt.month - forecasting_data["Sanction_Date"].dt.month)

forecasting_data = forecasting_data.merge(rating_count[["Origination_Rating", "Lambda"]], 
                                                    left_on="Origination_Rating", 
                                                    right_on="Origination_Rating")



# Getting the Outstanding balance at each reporting quarter
pmt =forecasting_data["PMT"]
monthly_rate = forecasting_data["Origination_Interest_Rate"]/12
t = forecasting_data["Tenure"]
n = forecasting_data["MOB"]

#Default Setting The O/s balance value for all loan types as 0 : 
forecasting_data["Outstanding_Balance"] = 0.0 # It makes sure that revolving loans

# 1 : Amortising Loans Outstanding balance Computation : 

# FOR NOW DECREASING O/S BALANCE IS DONE, NEED TO ADJUST FOR DEFAULTS, i.e, ADDING INTEREST IN CASE OF DEFAULTS 

forecasting_data.loc[is_amortising_loan,"Outstanding_Balance"]=(pmt*((1-(1+monthly_rate)**-(t-n))/(monthly_rate)))
                                                        
# Outstanding balance would be treated as "EAD" for amortising, revolving and bullet loans

# 2 : Bullet Loans Outstanding Balance Cmputation : 

is_bullet_loan = forecasting_data["Type_of_Loan"].isin(["Bullet"])
forecasting_data.loc[is_bullet_loan,"Outstanding_Balance"] =forecasting_data["Sanction_Amount (in USD)"]

# 3. Revolving Loans : --> O/s Balance would be the Drawn balance, EAD amount would be computed in a seperate column. 
revolving_low = 0.5 # 50%
revolving_high = 0.95 # 95%

rand_ = forecasting_data.groupby("Customer_ID")["Customer_ID"].transform(lambda x : np.random.uniform(low=revolving_low, high=revolving_high, size = len(x)))

is_revolving_loan = forecasting_data["Type_of_Loan"].isin(["Revolving"])

forecasting_data.loc[is_revolving_loan,"Outstanding_Balance"] = rand_ * forecasting_data["Sanction_Amount (in USD)"]

#Getting the forecast of the Macroeconomic variable (Z)

forecasting_data = forecasting_data.merge(mev_forecast[["observation_date","Z_Base","Z_Optimistic","Z_Pessimistic"]], 
                                         left_on="Reporting_Date",
                                         right_on="observation_date",
                                         how = "left")



forecasting_data["Year_left"] = (forecasting_data["Tenure"] - forecasting_data["MOB"])/12

# ————————————————
# EIR Computation : 
# ————————————————

temp =[]
EIR_annual = []
outflow = []
inflow = []
EIR_Final =[]
comp_vals = []

for i in range(len(forecasting_data)):
    
    if forecasting_data.at[i,"Quarter_Index"]==0:
        total_periods = forecasting_data.at[i,"Total_Periods"]        
        if forecasting_data.at[i,"Type_of_Loan"]=="Amortising":
            
            outflow= (-forecasting_data.at[i,"Sanction_Amount (in USD)"])
            inflows = [forecasting_data.at[i,"PMT"]] * forecasting_data.at[i,"Tenure"]
            comp_vals = [outflow] + inflows
            start_date = forecasting_data.at[i,"Sanction_Date"].date() 
            start_date_offset = start_date + DateOffset(months= -1)
            end_date = forecasting_data.at[i,"Maturity_Date"].date()
            dates = pd.date_range(start=start_date, end=end_date,freq="ME")
            eir = pyxirr.xirr(dates,comp_vals)
            EIR_annual.append(eir)
            temp = [eir] * total_periods
            EIR_Final.append(temp)
            
        elif forecasting_data.at[i,"Type_of_Loan"]=="Bullet":
            
            outflow= (-forecasting_data.at[i,"Sanction_Amount (in USD)"])
            inflows = [(forecasting_data.at[i,"Origination_Interest_Rate"]/12)*-outflow] * forecasting_data.at[i,"Tenure"]
            comp_vals = [outflow] + inflows
            start_date = forecasting_data.at[i,"Sanction_Date"].date() 
            start_date_offset = start_date + DateOffset(months= -1)
            end_date = forecasting_data.at[i,"Maturity_Date"].date()
            dates = pd.date_range(start=start_date, end=end_date,freq="ME")
            eir = pyxirr.xirr(dates = dates,amounts= comp_vals)
            EIR_annual.append(eir)
            temp = [eir] * total_periods
            EIR_Final.append(temp)
        else: # Revolving Loans 
            
            eir = forecasting_data.at[i,"Origination_Interest_Rate"]
            EIR_annual.append(eir)
            temp = [eir] * total_periods
            EIR_Final.append(temp)
    

# ------- Setting the Effective Interest Rate : 
EIR_Final = [item for sublists in EIR_Final for item in sublists]
forecasting_data["EIR_Annual"] = EIR_Final

EIR_annual = forecasting_data["EIR_Annual"] 

EIR_quarterly = ((1+ EIR_annual)**(1/4)) -1 # Converting to Quarterly Rate

forecasting_data["EIR_Compounding_Factor"] = (1+EIR_quarterly)**forecasting_data["Quarter_Index"]

forecasting_data

# Transition Matrix :

In [ ]:

raw_transition_matrix = pd.DataFrame(index=rating_order)

#                                  ["AAA" , "AA"  , "A"  , "BBB" , "BB"  , "B"   , "CCC" ,  "D"]
raw_transition_matrix["AAA"]    = [86.99 , 0.50  ,0.03  , 0.01  , 0.01  , 0.00  , 0.00   ,  0.00 ]
raw_transition_matrix["AA"]     = [9.12  , 87.06 ,1.69  , 0.09  , 0.03  , 0.02  , 0.00   ,  0.00 ]
raw_transition_matrix["A"]      = [0.53  , 7.85  ,88.17 , 3.42  , 0.11  , 0.08  , 0.11   ,  0.00 ]
raw_transition_matrix["BBB"]    = [0.05  , 0.49  ,5.16  , 86.04 , 4.83  , 0.17  , 0.20   ,  0.00 ]
raw_transition_matrix["BB"]     = [0.08  , 0.05  ,0.29  , 3.62  , 77.50 , 4.93  , 0.59   ,  5.0  ]
raw_transition_matrix["B"]      = [0.03  , 0.06  ,0.12  , 0.46  , 6.65  , 74.53 , 13.21  ,  7.0  ]
raw_transition_matrix["CCC"]    = [0.05  , 0.02  ,0.02  , 0.11  , 0.55  , 4.42  , 43.51  ,  35.0 ]
raw_transition_matrix["D"]      = [0.00  , 0.02  ,0.06  , 0.17  , 0.65  , 3.44  ,26.89   ,  45.0 ]
raw_transition_matrix["NR"]     = [3.15  , 3.94  ,4.48  , 6.10  , 9.67  , 12.41 , 15.50  ,  8.0  ]

console.print(" -------------- Raw yearly Rating Transition Matrix ---------------",style="bold")
display(raw_transition_matrix)

raw_transition_matrix= raw_transition_matrix / 100
raw_transition_matrix["1_NR"] = 1- raw_transition_matrix["NR"]
# Adjusting the transition matrix probabilities by removing the "NR" category
annual_transition_matrix= raw_transition_matrix.iloc[:,:-2].divide(raw_transition_matrix["1_NR"],axis=0)

# Adjusting the prbabilites for each row such that the sum of row is 100%
row_sums = annual_transition_matrix.sum(axis=1)
annual_transition_matrix = annual_transition_matrix.divide(row_sums,axis=0)

# ——————————————————————————————————
# The Optimised Transition Matrix : 
# ——————————————————————————————————

console.print(" --------------  Adjusted Cumulative Yearly TTC Rating Transiton Matrix ---------------",style="bold")
cumulative_transition_matrix = annual_transition_matrix.cumsum(axis=1)
display(cumulative_transition_matrix)

# The TTC yearly matrix needs to be converted to a quarterly TTC matrix : 

pd_map = rating_count["PD_estimation_correlated (%)"].to_list()

# Function Call : 
annual_transition_matrix = rk.matrix_optimisation(annual_transition_matrix,total_investment_grades_cols = 3,fix_default_rates = True, default_rates = pd_map)

console.print("--------------- Final Optimised Cumulative yearly TTC Rating Transition Matrix -------------- ",style="bold")
display(annual_transition_matrix.cumsum(axis=1))
console.print("--------------- Final Optimised yearly TTC Rating Transition Matrix -------------- ",style="bold")
display(annual_transition_matrix.style.format("{:.4f}"))

quarterly_matrix = fractional_matrix_power(annual_transition_matrix, 0.25)

clean_quarterly_matrix = np.real(quarterly_matrix)
clean_quarterly_matrix = clean_quarterly_matrix.clip(min=0)


row_sums = clean_quarterly_matrix.sum(axis=1, keepdims=True)
normalized_quarterly_matrix = clean_quarterly_matrix / row_sums

transition_matrix = pd.DataFrame(normalized_quarterly_matrix, index=rating_order, columns=rating_order)

cumulative_transition_matrix = transition_matrix.cumsum(axis=1)

console.print("--------------- Final Optimised Quarterly Cumulative TTC Rating Transition Matrix -------------- ",style="bold")

display(cumulative_transition_matrix)


# ———————— DPD TRANSITION MATRIX : 

dpd_tm = np.array([
    [0.90,0.05,0.04,0.01],
    [0.20,0.65,0.05,0.10],
    [0.10,0.05,0.80,0.05],
    [0.01,0.01,0.03,0.95]
])


dpd_transition_matrix = pd.DataFrame(dpd_tm, index = buckets, columns = buckets)

# ————————————————————————
# Optimised DPD Matrix :
# ————————————————————————
dpd_transition_matrix = rk.matrix_optimisation(dpd_transition_matrix)

cumulative_dpd_transition_matrix = dpd_transition_matrix.cumsum(axis=1)

console.print("--------------- Raw Quarterly TTC DPD Transition Matrix -------------- ",style="bold")
display(pd.DataFrame(dpd_tm,index = buckets, columns = buckets))

console.print("--------------- Final Optimised Quarterly Cumulative TTC DPD Transition Matrix -------------- ",style="bold")
display(cumulative_dpd_transition_matrix)

#### Computing the PIT_Tansition Matrix of Rating and DPD for each reported quarter : 

In [ ]:


"""
    There are 12 reported quarters spanning from March 31st 2023 to December 31st 2025.
    Furthermore, there are 3 cases being considered : Base, Optimistic, Pessimistic, thus
    there will be a total of 36 pit_transition_matrix. 
    For computational purposes, a flat 12% correlation is considered.
"""

# Lower and Upper bounds for DPD Transition Matrix : 

# 1. Lower Bound DPD :

lower_bound_dpd = pd.DataFrame(index = dpd_transition_matrix.index, columns=dpd_transition_matrix.columns)

lower_bound_dpd = dpd_transition_matrix.iloc[:,::-1].cumsum(axis=1).iloc[:,::-1]
lower_bound_dpd = lower_bound_dpd.clip(lower=0.0, upper = 1.0)

lower_bound_dpd.iloc[:,:-1] = stats.norm.ppf(lower_bound_dpd.iloc[:,1:])
lower_bound_dpd.iloc[:,-1] = -10_000

# 2. Upper Bound DPD : 

upper_bound_dpd = lower_bound_dpd.shift(axis=1).fillna(10_000)


# Lower and Upper bounds for Rating Transition Matrix : 

# 1. Lower Bound Rating :

lower_bound_tm = pd.DataFrame(index = cumulative_transition_matrix.index, columns=cumulative_transition_matrix.columns)

lower_bound_tm = transition_matrix.iloc[:,::-1].cumsum(axis=1).iloc[:,::-1]
lower_bound_tm = lower_bound_tm.clip(lower=0.0, upper = 1.0)

# Taking the inverse of the standard normal distribution of the lower_bound_tm :
lower_bound_tm.iloc[:,:-1] = stats.norm.ppf(lower_bound_tm.iloc[:,1:])

lower_bound_tm = lower_bound_tm.fillna(1e-07)

lower_bound_tm.iloc[:,-1] = -10_000

# 2. Upper Bound :

upper_bound_tm = lower_bound_tm.shift(axis=1).fillna(10_000).copy()

# ---Empty Point in time rating transition matrix -------
pit_transition_matrix = pd.DataFrame(index = cumulative_transition_matrix.index, columns = cumulative_transition_matrix.columns)

# ======================= Computation of PIT-Transition Matrices (Rating and DPD) :

scenario_count = 0
total_count = 0

pit_tm_dict = {}
pit_dpd_tm_dict = {}

PIT_tm_dict_non_cum = {} # Used for non cumulative transition matrix
for s in range(len(Z_scenarios)):
    
    for q in range(len(reporting_dates_list)):
        
        Z_val = mev_forecast.loc[mev_forecast["observation_date"]==reporting_dates_list[q],Z_scenarios[s]].values[0]
        
        pit_vals = stats.norm.cdf((upper_bound_tm - np.sqrt(rho) * Z_val)/np.sqrt(1-rho)) - \
                   stats.norm.cdf((lower_bound_tm - np.sqrt(rho) * Z_val)/np.sqrt(1-rho))
        
        # Coversion to the dataframe : 
        pit_transition_matrix = pd.DataFrame(pit_vals.cumsum(axis=1), index= cumulative_transition_matrix.index, columns= cumulative_transition_matrix.columns)
        #pit_transition_matrix = pd.DataFrame(pit_vals, index= cumulative_transition_matrix.index, columns= cumulative_transition_matrix.columns)
        dictionary_key_title = f"{reporting_dates_list[q]}_{Z_scenarios[s].removeprefix('Z_')}"
        
        # Inserting the transition matrix onto a Dictionary : 
        pit_tm_dict.update({dictionary_key_title : pit_transition_matrix}) 

        # -----------------------------------------------
        # Non Cumulative Transition matrix : 
        transition_matrix_non_cumulative = pd.DataFrame(pit_vals, index= cumulative_transition_matrix.index, columns= cumulative_transition_matrix.columns)
        PIT_tm_dict_non_cum.update({dictionary_key_title : transition_matrix_non_cumulative})
        # -----------------------------------------------
        
        print(f"\033[1mCumulative PIT Rating Transition Matrix for date: {reporting_dates_list[q]}; Scenario: {Z_scenarios[s].removeprefix('Z_')} and a Z of {Z_val:.5f}\033[0m")
        display(pit_transition_matrix)
        
        # ----------- PIT - DPD Transition Matrix : 
        
        dpd_vals = stats.norm.cdf((upper_bound_dpd - np.sqrt(rho)*Z_val)/np.sqrt(1-rho)) - \
        stats.norm.cdf((lower_bound_dpd - np.sqrt(rho)*Z_val)/np.sqrt(1-rho))
        
        # Conversion to DataFrame : 
        
        pit_dpd_tm = pd.DataFrame(dpd_vals.cumsum(axis=1), index=cumulative_dpd_transition_matrix.index, columns= cumulative_dpd_transition_matrix.columns)
        dictionary_key_title = f"{reporting_dates_list[q]}_{Z_scenarios[s].removeprefix('Z_')}"
        
        # Inserting the DataFrame into a Dictionary : 
        
        pit_dpd_tm_dict.update({dictionary_key_title :pit_dpd_tm})
        
        print(f"\033[1mCumulative DPD Transition Matrix for Date : {reporting_dates_list[q]} ; Scenario : {Z_scenarios[s].removeprefix('Z_')} and a Z of {Z_val:.5f}\033[0m")
        display(pit_dpd_tm)
        
        total_count+=1
        

####  Illustrative Example of Point in time CCF :

- Actual PIT CCF/ LGD is computed within the main function call.

In [ ]:
"""
The following is an iillustration of CCF values over the reported period for Retail and SME Revolving loans, where a 
flat CCF of 75% is selected and adjusted as per macroeconomic condition.

"""
def CCF_Computation(ccf=0.75):
    rho = 0.12
    
    for s in scen:
        
        mev_forecast[f"CCF_{s}"] = stats.norm.cdf((stats.norm.ppf(ccf)-np.sqrt(rho)*mev_forecast[f"Z_{s}"])/np.sqrt(1-rho))
    
    fig, ax  = plt.subplots(1,1, figsize=(9,4))
    
    for idx, s in enumerate(scen):
        ax.plot(mev_forecast["observation_date"],mev_forecast[f"CCF_{s}"],label= f"CCF_{s}",linestyle="--", color=ScenarioColorCode[idx])
    
    ax.set_ylabel("Point in Time - CCF")
    ax.set_xlabel("Reporting Quarters")
    ax.set_title("Point-in-time Credit Conversion Factor (CCF)")
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.axhline(y=1, linestyle="--", color="k")
    ax.scatter(y =ccf, x = mev_forecast["observation_date"][0], color="gold", label="Starting CCF Value")
    plt.legend(bbox_to_anchor=(1.05,1))
    plt.show()
    display(mev_forecast.tail(13))

widgets.interact(CCF_Computation, ccf=widgets.FloatSlider(value=0.75, min=0.3,max=0.95,step=0.05,description="CCF Value"))


# Markov Chain Simulation

In [ ]:
importlib.reload(rk)

"""
In the below dictionary, all the keys represents the column names that are within the dataset on which analysis is being done and
all the values represent the names that the `ECL_sim` function takes. Make sure to adjust the column names in the Dataset as per that to avoid any errors.
The most important column, which must match to what's in the function are :

1. "Quarter_Index"
2. "Q_Count_for-Rating_Est."
3. "Reporting_Date"
4. "Origination_Rating"
5. "Quarter_Count"
6. "Type_of_Loan"
7. "Origination_Interest_Rate"
8. "Tenure"
9. "MOB"
10. "Segment"
11. "Customer_ID"
12. "Outstanding_Balance"
13. "Sanction_Amount (in USD)"
14. "PMT_New"

"""

Forecasting_data_ColumnList = {'Customer_ID': 'Customer_ID',                              # 1
                               'Segment': 'Segment',                                      # 2
                               'Sector': 'Sector',                                        # 3
                               'Sanction_Amount (in USD)': 'Sanction_Amount (in USD)',    # 4
                               'Outstanding_Balance': 'Outstanding_Balance',              # 5
                               'Type_of_Loan': 'Type_of_Loan',                            # 6
                               'Tenure': 'Tenure',                                        # 7
                               'Maturity_Date': 'Maturity_Date',                          # 8
                               'Loan_Duration_Flag': 'Loan_Duration_Flag',                # 9
                               'Sanction_Date': 'Sanction_Date',                          # 10
                               'Origination_Rating': 'Origination_Rating',                # 11
                               'Origination_Interest_Rate': 'Origination_Interest_Rate',  # 12
                               'Secured_Unsecured_Flag': 'Secured_Unsecured_Flag',        # 13
                               'Collateral_Type': 'Collateral_Type',                      # 14
                               'Collateral_Value': 'Collateral_Value',                    # 15
                               'Reporting_Date': 'Reporting_Date',                        # 16
                               'MOB': 'MOB',                                              # 17
                               'end_date_amort': 'end_date_amort',                        # 18
                               'start_date_clipped': 'start_date_clipped',                # 19
                               'end_date': 'end_date',                                    # 20
                               'reporting_end_date': 'reporting_end_date',                # 21
                               'Quarter_Count': 'Quarter_Count',                          # 22
                               'Total_Periods': 'Total_Periods',                          # 23
                               'Quarter_Index': 'Quarter_Index',                          # 24
                               'Q_Count_for_Rating_Est.': 'Q_Count_for_Rating_Est.',      # 25
                               'PMT': 'PMT',                                              # 26
                               'Lambda': 'Lambda',                                        # 27
                               'observation_date': 'observation_date',                    # 28
                               'Z_Base': 'Z_Base',                                        # 29
                               'Z_Optimistic': 'Z_Optimistic',                            # 30
                               'Z_Pessimistic': 'Z_Pessimistic',                          # 31
                               'Year_left': 'Year_left',                                  # 32 
                               'EIR_Annual': 'EIR_Annual',                                # 33
                               'EIR_Compounding_Factor': 'EIR_Compounding_Factor'         # 34  
                              }

forecasting_data = forecasting_data.rename(columns= Forecasting_data_ColumnList)

# ——— Getting the required arguments for the function : 

# 1 :
segment_chargeoff_caps = {"Retail" : 2,     "SME" : 2,        # Cap of 180 days or 2 quarters -> Change if needed
                          "Corporate" : 2,  "Commercial" : 2} # Cap of 180 days or 2 quarters -> Change if needed

# 2 : 
lambda_map = rating_count.set_index("Origination_Rating")["Lambda"]

buckets_dict = {val : idx for idx, val in enumerate(buckets)} # Creating the dictionary 

pd_map = rating_count.set_index("Origination_Rating")["PD_estimation_correlated (%)"]

# 3 :

# ————————————————————————
#  MULTIPROCESSING

# Step 1 : Getting the Number of iterations and creating bins based on the number of CPU Core's we have
# ————————————————————————

# ————————————————————————
# Iteration Assignment : 
# ———————————————————————

iterations = 10

weights = [0.4,0.3,0.3] # Weights given to each scenario. The weights must follow the order of the "scen" list


ScenarioWeights = dict(zip(scen,weights))

if __name__ == "__main__":

    total_logical_cpu_cores = os.cpu_count()
    n_workers = os.cpu_count() - 5

    if iterations < n_workers:
        #print(f"Few Runs compared to workers that would be assigned : Workers assigned : {n_workers}; total iterations : {iterations}")
        n_workers = iterations
    
    ValueSet =[] # Creates a list of items with start and end value of iteration to be assigned to each worker. Stored as : [start, end, start, end,....]
    startVal = 0
    EndVal = 0
    
    tasks = [] # A list of groups of iterations being send to each CPU core.
    
    # ————————————————————————
    # Step 2 : Assigning the iterations to each CPU's Core : 
    # ————————————————————————
    
    base = iterations // n_workers
    remainder = iterations % n_workers
    
    for i in range(n_workers):
        Bin_size = base + (1 if i < remainder else 0)
        EndVal = startVal + Bin_size
        tasks.append((startVal, EndVal))
        startVal = EndVal
    
    books_MC = {}
    Iteration_results_ECL = pd.DataFrame()
    #historical_dpd_last = None
    
    # ————————————————————————
    # Function Call :
    # ————————————————————————
    
    return_data = True
    
    #for i in range(10):
    #    start_time = time.perf_counter()
    
    start_time = time.perf_counter()
    
    try :
        with ProcessPoolExecutor(max_workers=n_workers, max_tasks_per_child = 100) as ex:
            futures = [ex.submit(rk.ECL_sim,
                                dataset=forecasting_data,
                                iteration_start=iteration_start, iteration_end=iteration_stop, iterations=iterations,
                                CCF=CCF, LGD=LGD, scen=scen, pd_map=pd_map, lambda_map=lambda_map,
                                buckets_dict=buckets_dict, ScenarioWeights=ScenarioWeights,
                                cumulative_dpd_transition_matrix=cumulative_dpd_transition_matrix,
                                cumulative_transition_matrix=cumulative_transition_matrix,
                                pit_dpd_tm_dict=pit_dpd_tm_dict, pit_tm_dict=pit_tm_dict,
                                buckets=buckets, rating_order=rating_order,return_result_data = return_data
                                )
                       for iteration_start, iteration_stop in tasks
                      ]
            
            for fut in futures:
                if return_data == True:
                    books, Iteration_results = fut.result()
                    books_MC.update(books)
                    books_MC = {key: books_MC[key] for key in sorted(books_MC, key = lambda x : int(x.split("_")[1]))}
                    Iteration_results_ECL = pd.concat([Iteration_results_ECL, Iteration_results], axis=0)
                else:
                    Iteration_results = fut.result()
                    Iteration_results_ECL = pd.concat([Iteration_results_ECL, Iteration_results], axis=0)
    except Exception:
        raise

    else:
        #timeVals.at[i,"Run"] = i
        #end_time = time.perf_counter()
        #timeVals.at[i,"Optimised_TimeTaken"] = end_time - start_time
        #timeVals.at[i, "Logical CPU Cores used"] = n_workers
        #timeVals.at[i, "Logical CPU Cores available"] = total_logical_cpu_cores
        end_time = time.perf_counter()
        print(f" Simulation Executed successfully!\n Total time taken: {end_time - start_time} seconds")#,style="bold green")
        

In [ ]:
print(f" Shape : {Iteration_results_ECL.shape}")
Iteration_results_ECL

In [ ]:
Iteration_results_ECL

In [ ]:
# ————————————————————
# ECL Convergence plot :
# ————————————————————

convergence_data = Iteration_results_ECL
# Uncomment the following to see the results of 800 Monte Carlo ECL simulations :

# convergence_data = pd.read_excel("Monte_Carlo_ECL_simulation_800.xlsx")



total_reporting_dates = len(reporting_dates_list)
n_iter = convergence_data.shape[0]//total_reporting_dates
ECL_col_list = convergence_data.columns.tolist()[1:]

def convergence_plot_ECL(ECL_col = ECL_col_list[-1]):
    nrows = total_reporting_dates // 2
    fig, ax = plt.subplots(nrows,2,figsize=(15,14))
    flatten_ax = ax.flatten()
    for idx, axes in enumerate(flatten_ax):
        f = convergence_data[ECL_col].to_numpy().reshape(n_iter, 12)[:, idx]   # last reporting date
        n = np.arange(2, n_iter + 1) # k
        sum_value = f.cumsum()[1:] / n # m
        se = np.array([f[:i].std(ddof=1) / np.sqrt(i) for i in n])
        axes.plot(n, sum_value)
        axes.set_title(f" Convergence plot for date : {reporting_dates_list[idx]}")
        axes.fill_between(n,sum_value - 1.96 * se, sum_value + 1.96 * se, alpha=0.3)
    fig.supxlabel("Number of Iterations")
    fig.supylabel("Weighted ECL value", x = 0.01)
    fig.suptitle(f"Convergence Plot for {ECL_col}", fontweight="bold", fontsize=12, color="#22252A")
    plt.tight_layout()
    
    # Saving the plot as a png file : 
    # plt.savefig("11_Monte Carlo convergence.png")
    
    plt.show()
    
widgets.interact(convergence_plot_ECL,
                 ECL_col = widgets.Select(options = ECL_col_list,value = ECL_col_list[-1]))

In [ ]:
# ————————————————————————————————————
# Merging the output dataframe (active set) with the input dataframe used for further analysis :

# Uncomment the followig if the return_history was set as True in the ECL_sim function.

# ————————————————————————————————————

final_active_book = forecasting_data.copy()

if isinstance(books_MC,dict) and len(books_MC) > 0 :
    for idx, s in enumerate(scen):
        
        active_book = books_MC["Iteration_0"][s]["active"]
    
        if idx > 0 and "Weighted_ECL" in active_book.columns :
            active_book = active_book.drop(columns=["Weighted_ECL"])
        
        final_active_book = pd.merge(final_active_book,
                                     active_book,
                                     on = ["Customer_ID","Reporting_Date"],
                                     how = "left")
                                 
    display(final_active_book)
    console.print("\nNote : The NaN values in the columns Stage_{s}, DPD Bucket_{s}, Estimated_Rating_{s} and ECL{s}, where s refers to scenarios,"
                  f" shows that for that particular scenario, the particular loan has defaulted and thus it's observations were moved to the charge_off book", style="bold")

In [ ]:
#%%time
# # ——————————————————
# # Un-Optimised Code (Used for Comparison of time taken
# # ——————————————————

# run_code = 10

# if not isinstance(timeVals, pd.DataFrame)
#     timeVals = pd.DataFrame()

# for i in range(run_code):
#     start_time = time.perf_counter()
#     vals = rk.ECL_sim(dataset=forecasting_data,
#                             iteration_start=0, iteration_end=35, iterations=35,
#                             CCF=CCF, LGD=LGD, scen=scen, pd_map=pd_map, lambda_map=lambda_map,
#                             buckets_dict=buckets_dict, ScenarioWeights=ScenarioWeights,
#                             cumulative_dpd_transition_matrix=cumulative_dpd_transition_matrix,
#                             cumulative_transition_matrix=cumulative_transition_matrix,
#                             pit_dpd_tm_dict=pit_dpd_tm_dict, pit_tm_dict=pit_tm_dict,
#                             buckets=buckets, rating_order=rating_order,
#                             )
#     end_time = time.perf_counter()
#     timeVals.at[i,"Run"] = i
#     timeVals.at[i,"Unoptimised_TimeTaken"] = end_time - start_time
#     print(f" Iteration {i} Completed ")
    
# timeVals["Reduction"] = abs((timeVals["Optimised_TimeTaken"] / timeVals["Unoptimised_TimeTaken"])-1)

# optimised_total = timeVals["Optimised_TimeTaken"].sum()
# unoptimised_total = timeVals["Unoptimised_TimeTaken"].sum()
# print(f" optimised_total:{optimised_total} seconds or {optimised_total/60} minutes \n"
#       f" unoptimised_total : {unoptimised_total} seconds or {unoptimised_total/60} minutes\n")

# ————————————————————
# Comparison Plot :
# ————————————————————
# fig, ax = plt.subplots(figsize=(15,5))

# plt.rcParams['axes.edgecolor'] = 'darkgrey'

# ax.bar(timeVals["Run"], timeVals["Unoptimised_TimeTaken"], width = 0.6,label = "Unoptimised_TimeTaken", color="#2C3E50")
# ax.bar(timeVals["Run"], timeVals["Optimised_TimeTaken"], width= 0.45,label = "Optimised_TimeTaken", color="darkgrey")
# ax.set_ylabel("Time Taken (in seconds)", color="black", fontweight="bold")
# ax.set_xlabel("Number of runs", fontweight="bold", color="black")

# ax2 = ax.twinx()

# ax2.plot(timeVals["Reduction"],color="saddlebrown",linestyle="--", label="% reduction", linewidth = 2, marker="o")
# ax2.yaxis.set_major_formatter(PercentFormatter(xmax=1))
# ax2.set_ylim(bottom=0.72,top=0.76,auto=True)
# ax2.set_ylabel("Performance Improvement", color="black", fontweight="bold")
# ax2.set_ylim(top=0.765)
# ax2.axhline(y = timeVals["Reduction"].mean(), label="Average reduction", color="black", alpha=1,linewidth=2.5, linestyle="dotted")

# handles1, labels1 = ax.get_legend_handles_labels()
# handles2, labels2 = ax2.get_legend_handles_labels()

# ax.legend(handles1 + handles2, labels1 + labels2, loc="upper left", 
#           facecolor='white', edgecolor='#D3D3D3', ncols=4, frameon=True)

# plt.suptitle("Performance Benchmarking: \nMultiprocessing vs. Sequential Execution of ECL Monte Carlo Simulations*",
#              x= 0.1,horizontalalignment="left", verticalalignment="top", fontweight="bold",fontsize=10,style="italic", color="#22252A")

# fig.text(0.05, 0.01, "*10 independent runs of 35 Monte Carlo simulations were performed. The runs were performed on the same input data and values.", 
#          fontsize=8.5, style='italic', color='#22252A', ha="left")

# #plt.savefig("10_Performance Benchmark.png", dpi= 400) 
# fig.tight_layout()
# plt.show()

In [ ]:
# ——————————————————
# Checking if the DPD and Ratings show a correlation.
# Given the Ratings are DPD are categories, we used the chi square test to check if relationship exists.
# The result of the output shows the p value for each Rating x DPD. If p-value <= 0.05, we conclude that the relationship exists.
# ——————————————————

file_ = "chi_square test_Copulas.md"

with open(file_, "a")as f:
    f.write("\n Chi square Test for testing Correlation between categorical variables - After Copula is induced \n")
#try :
if books_MC!= {}:
    for s in scen :
        checkData = pd.merge(books_MC["Iteration_1"][f"{s}"]["active"],
             books_MC["Iteration_0"][f"{s}"]["charge_off"],how="outer",
             left_on=books_MC["Iteration_1"][f"{s}"]["active"].columns.tolist(),
             right_on=books_MC["Iteration_1"][f"{s}"]["charge_off"].columns.tolist())
        
        crosstab_data = pd.crosstab(checkData[f"Estimated_Rating_{s}"], checkData[f"DPD Bucket_{s}"])[buckets] # the [buckets] reorders the columns. 
        chi_test_output = pd.DataFrame(stats.chi2_contingency(crosstab_data)[3])
        
        cat_var1 = tuple(rating_order)
        cat_var2 = tuple(buckets)
        cat_var_prod = list(product(cat_var1,cat_var2, repeat = 1))
        result = []
        for i in cat_var_prod:
            if i[0] != i[1]:
                result.append((i[0],i[1],list(stats.chi2_contingency(pd.crosstab(
                                        checkData[f"Estimated_Rating_{s}"], checkData[f"DPD Bucket_{s}"])))[1]))
        result
        chi_test_output = pd.DataFrame(result, columns = ["Rating", "DPD", 
                                                               "coeff"])
        ## Using pivot function to convert the above DataFrame into a crosstab
        print(f" Scenario : {s}\n Output Chi-square test to check if DPD and Ratings show a relation :\n"
             f" Reject the Null if value < 0.05\n") 
        outputfile = chi_test_output.pivot(index="Rating", columns="DPD", values="coeff")

        # Calculate Cramer's V (returns a value between 0 and 1)
        cramers_v = association(crosstab_data, method="cramer")
        print(f"Cramer's V (Correlation Strength): {cramers_v:.4f}")

        display(outputfile)
        print(f"\n")
        with open(file_, "a") as f:
            f.write(f" Scenario : {s}\n Output Chi-square test to check if DPD and Ratings show a relation :\n"
             f" Reject the Null if value < 0.05\n")
            f.write(f"Cramer's V (Correlation Strength): {cramers_v:.4f}")
            f.write(outputfile.to_markdown(floatfmt=".4f"))
#except :
#    print("books_MC not defined. Make sure that the `return_result_data` argument in the ECL_sim function was set as True.")

In [ ]:
# Define dictionaries that explicitly declare the true ordinal order
# Lower number = safer/lower risk (or vice versa, just keep it uniform)
rating_mapping = {label: idx for idx, label in enumerate(rating_order)}
dpd_mapping = {label: idx for idx, label in enumerate(buckets)}

for s in scen:
    # 1. Base Merge
    checkData = pd.merge(
        books_MC["Iteration_1"][f"{s}"]["active"],
        books_MC["Iteration_0"][f"{s}"]["charge_off"],
        how="outer",
        left_on=books_MC["Iteration_1"][f"{s}"]["active"].columns.tolist(),
        right_on=books_MC["Iteration_1"][f"{s}"]["charge_off"].columns.tolist()
    )
    
    # 2. Map categorical strings to ordered integers
    # Dropping missing rows is crucial as NaN will break rank-based math
    temp_df = checkData[[f"Estimated_Rating_{s}", f"DPD Bucket_{s}"]].dropna()
    
    x_mapped = temp_df[f"Estimated_Rating_{s}"].map(rating_mapping)
    y_mapped = temp_df[f"DPD Bucket_{s}"].map(dpd_mapping)
    
    # 3. Compute Spearman Rank Correlation directly on raw ranks
    spearman_corr, spearman_p = stats.spearmanr(x_mapped, y_mapped)
    
    # 4. Compute Kendall's Tau-b (preferred over Spearman if there are heavy ties)
    kendall_corr, kendall_p = stats.kendalltau(x_mapped, y_mapped)
    
    # 5. Output Results
    print(f"==================================================")
    print(f" Scenario : {s} (Copula Induced Correlation Evaluation)")
    print(f"==================================================")
    print(f" Spearman Rank Rho : {spearman_corr:.4f} (p-val: {spearman_p:.4f})")
    print(f" Kendall's Tau-b   : {kendall_corr:.4f} (p-val: {kendall_p:.4f})\n")

# ECL Visualisations : 

In [ ]:
# Uncomment the following to see the results of 800 Monte Carlo ECL simulations :
# Iteration_results_ECL = pd.read_excel("Monte_Carlo_ECL_simulation_800.xlsx",index_col=0)

def dist_plots(reporting_date="2025-12-31"):
    
    numberofIterations = Iteration_results_ECL.index.value_counts().iloc[0]
    if numberofIterations < 2 : 
         console.print(f" Please run a higher number of iterations to get a better estimate",style="bold yellow")
    else:
        fig,ax = plt.subplots(nrows=4, ncols=2, sharey = "row", gridspec_kw={"width_ratios":[2,3]}, figsize=(20,30))
        plt.subplots_adjust(wspace=0.01, hspace=0.6)
        
        data = Iteration_results_ECL[Iteration_results_ECL.index==reporting_date].reset_index(drop=True)
        data.rename(columns={"ECL_Final" : "ECL_Weighted"}, inplace=True)
         
        colorList = ["#2B5278", "#047857", "indianred", "darkslategrey"]
        
        for idx, s in enumerate(data.columns) :
            sns.lineplot(data[s],ax=ax[idx,0], color = colorList[idx])
            ax[idx,0].set_xlabel("Number of Iterations")
            ax[idx,0].set_ylabel("ECL Value")
            ax2 = ax[idx,1].twiny()
            
            sns.kdeplot(y=data[s], ax=ax2, color=colorList[idx]) # ax[idx,1]
            ax[idx,1].hist(data[s],density=True,bins="fd", ec="w", fc= colorList[idx], orientation = "horizontal", alpha = 1)
            
            percentile_values = {}
            for p, c in zip([99.9, 95, 2.5], ["red", "black", "blue"]):
                key = f"{p} percentile"
                percentile_values[key]= np.percentile(data[s], p)
                [axes.axhline(percentile_values[key], color=c, linestyle="--", label =key) for axes in ax[idx, :]]        
                
            ax[idx,0].spines['right'].set_visible(False)
            ax[idx,1].set_xlabel("Histogram counts")
            ax2.set_xlabel("KDE Density")
            text_str = "\n".join([f"{k} ECL: ${ v:,.2f}" for k, v in percentile_values.items()])
            ax[idx, 1].annotate(text_str, xy=(0.6, 0.8), xycoords="axes fraction", fontsize=12)
            ax[idx, 0].set_title(f"\nScenario Analysis: {s.upper()}\n", x=1.0, y=1.02, fontsize=13, fontweight="bold")
            mean_val = data[s].mean()
            print(f"mean value for {s} -> ${mean_val:,.2f}")
            # print(f"Capital for {s} Scenario : ${percentile_values["99.9 percentile"] - mean_val:,.2f}\n")
        handles, labels = ax[idx, 0].get_legend_handles_labels()
        fig.text(0.12, 0.09, f"Red dashed line : 99.9 percentile, black dashed line : 95th and blue dashed line : 2.5th percentile ECL values", 
         fontweight="bold",fontsize=10, style='italic', color='#22252A', ha="left")
        
        # Uncomment to save as an image : 
        # plt.savefig("ECL Distribution Chart.png", dpi=300, bbox_inches="tight", pad_inches=0.1)
        
        plt.show()
widgets.interact(dist_plots, reporting_date=widgets.Select(options = reporting_dates_list, description="Reporting Date", style={"description_width":"120px"}))

In [ ]:
# ——————————————————
# 1. ECL Bar Chart :
# ——————————————————

def ecl_reporting_date(dataset=None, reporting_date="2023-12-31"):

    if dataset is None:
        raise ValueError("No Data provided")

    colorlist = ["#2B5B84", "#27AE60", "#C0392B"]
    
    if len(colorlist) < len(scen):
        raise ValueError("The number of colors provided do not match the scenario count.")

    ECL_comp = dataset[dataset["Reporting_Date"] == reporting_date]

    ECL_values = {s: ECL_comp[f"ECL_{s}"].sum() for s in scen}

    table_1 = Table(title="ECL values")
    table_1.add_column("Scenario", style="Bold Black")
    table_1.add_column("Value", style="cyan", justify="right")
    
    for s in scen:
        table_1.add_row(s, f"{ECL_values[s]:,.4f}")

    Final_ECL = sum(ECL_values[s] * ScenarioWeights[s] for s in scen)

    table_2 = Table(title=f"Final ECL as on date: {reporting_date}", style="Bold")
    table_2.add_column("Date", style="Bold")
    table_2.add_column("ECL Value in USD", style="cyan")
    table_2.add_row(reporting_date, f"{Final_ECL:,.2f}")

    grid = Table.grid(padding=4) # Add horizontal padding between tables
    grid.add_column()
    grid.add_column()
    
    grid.add_row(table_1, table_2)
    console.print(grid)
    
    plot_data = pd.DataFrame({"Scenario": [f"ECL_{s}" for s in scen] + ["Weighted_ECL"],
                              "ECL": [ECL_values[s] for s in scen] + [Final_ECL]})
    
    fig, ax = plt.subplots(figsize=(6, 4))

    palette = colorlist[: len(scen)] + ["#D4AC0D"]
    
    sns.barplot(data=plot_data, x="Scenario", y="ECL", palette=palette, edgecolor="black", width=0.3, hue= "Scenario", ax=ax)
    
    ax.set_title(f"Expected Credit Loss by Scenario as on {reporting_date}", fontsize=12, fontweight='semibold', color='#475569')
    ax.set_ylabel("ECL Amount (in USD hundred million)",fontsize=10,fontweight='bold', color='#475569')
    plt.tight_layout()

    minimum_val = min(ECL_values.values()) * 0.1

    def scaled(x, pos):
        return str(round(x / 1e8, 1))

    ax.yaxis.set_major_formatter(scaled)
    ax.set_xlabel("")
    ax.yaxis.set_minor_formatter(NullFormatter())
    sns.despine()

    # Uncomment to save the plot as an image :
    # fig.savefig("1_ECL Bar plots.png", bbox_inches="tight", dpi=400)
    
    plt.show()


# ————————————————————————————————————————————————
# 2 : Rating Distribution Bar Chart : 
# ————————————————————————————————————————————————

def rating_dist_rep_date(dataset = None, reporting_date="2023-12-31"):
    
    reporting_date = reporting_date
    
    seg_data = dataset[dataset["Reporting_Date"]==reporting_date]
    
    orig_counts = pd.DataFrame(seg_data["Origination_Rating"].value_counts().reindex(rating_order, fill_value=0))
    orig_counts.rename(columns={"count":"Origination"},inplace=True)
    
    base_counts = pd.DataFrame(seg_data["Estimated_Rating_Base"].value_counts().reindex(rating_order))
    base_counts.rename(columns={"count":"Base_Scenario"},inplace=True)
                            
    optimistic_counts = pd.DataFrame(seg_data["Estimated_Rating_Optimistic"].value_counts().reindex(rating_order))
    optimistic_counts.rename(columns={"count":"optimistic_Scenario"},inplace=True)
    
    pessimistic_counts = pd.DataFrame(seg_data["Estimated_Rating_Pessimistic"].value_counts().reindex(rating_order))
    pessimistic_counts.rename(columns={"count":"Pessimistic_Scenario"},inplace=True)
    
    
    comparison_data = pd.concat([orig_counts,base_counts, optimistic_counts, pessimistic_counts],axis=1)
    
    console.print(f"Rating Migration under Different Scenarios as on {reporting_date}",style="Bold")
    
    #sns.barplot(data=comparison_data)
    ax = comparison_data.plot(kind="bar", color=["khaki","navy","darkseagreen","indianred"], width=0.8,figsize=(12,6))
    ax.set_xlabel("Credit Ratings",fontsize=10,fontweight='bold', color='#475569')
    ax.set_ylabel("Number of Borrowers",fontsize=10,fontweight='bold', color='#475569')
    plt.legend(bbox_to_anchor=(1,1))
    ax.grid(alpha=0.3,axis="y")

    ax.spines[["top", "right"]].set_visible(False)
    
    # Uncomment to save the plot as an image :
    #plt.savefig("2_Rating Distribution_Q12025.png", bbox_inches="tight", dpi=400)
    
    plt.show()
    
    display(comparison_data)
    console.print(f" Base Scenario Total : {comparison_data['Base_Scenario'].sum()}\n"
          f" Optimistic Scenario Total : {comparison_data['optimistic_Scenario'].sum():.0f}\n"
          f" Pessimistic Scenario Total : {comparison_data['Pessimistic_Scenario'].sum()}")
 #   console.print("The count in each rating grade on a specific reporting date represents the borrowers before the loan book was segregated into Active loans and Chargeoff loans", style="Red bold")


# ————————————————————————————————————————————————
# 3. Credit Rating Migration Plot (Sankey Diagram) : 
# ————————————————————————————————————————————————

def credit_migration_diagram(dataset = None, reporting_date="2023-03-31",scenario="Base"):
    
    segment_order = ["Commercial","Corporate", "Retail", "SME"]
    
    label = rating_order + segment_order + rating_order
    
    seg_data = dataset[dataset["Reporting_Date"]==reporting_date]
    df1= seg_data.groupby(["Origination_Rating","Segment"])["Customer_ID"].count().reset_index()
    df1["Origination_Rating"] = pd.Categorical(df1["Origination_Rating"], categories = rating_order, ordered = True)
    df1 = df1.sort_values(by=["Origination_Rating","Segment"]).reset_index(drop=True)
    
    total = df1["Customer_ID"].sum() # Total number of Customers
    
    df1.columns = ["source","target","value"]
    df1["source"] = df1["source"].astype(str)
    df1["source"] = df1["source"] + "(Sanct.)" # S for Sanction Rating
    
    col_name = "Estimated_Rating_"+ scenario
    
    df2 = seg_data.groupby(["Segment",col_name])["Customer_ID"].count().reset_index()
    df2[col_name] = pd.Categorical(df2[col_name],categories=rating_order,ordered=True)
    df2 = df2.sort_values(by=["Segment", col_name]).reset_index(drop=True)
    
    df2.columns = ["source","target","value"]
    df2["target"] = df2["target"].astype(str)
    
    df2["target"] = df2["target"] + "(Report.)" 
    
    total_2 = df2["value"].sum()
    
    merged_df = pd.concat([df1,df2],axis=0)
    
    unique_source_target =  pd.unique(merged_df[["source","target"]].values.ravel("k"))
    
    mapping_dict = {k:v for v,k in enumerate(unique_source_target)}
    
    merged_df["source"] = merged_df["source"].map(mapping_dict)
    merged_df["target"] = merged_df["target"].map(mapping_dict)

    # used to create a list for links color: 
    reverse_mapping_dict = {k:v for k,v in enumerate(unique_source_target)}
    #print(f" Reverse reverse_mapping_dict : {reverse_mapping_dict}")
    
    merged_df_temp = merged_df.copy()
    merged_df_temp["source_ac"] = merged_df_temp["source"].map(reverse_mapping_dict)
    merged_df_temp["target_ac"] = merged_df_temp["target"].map(reverse_mapping_dict)
    #display(merged_df_temp)
    
    linkcolorList = np.where(((merged_df_temp["source_ac"]=="AAA(Sanct.)") | (merged_df_temp["source_ac"]=="AA(Sanct.)") | (merged_df_temp["source_ac"]=="A(Sanct.)")|
                             (merged_df_temp["target_ac"]=="AAA(Report.)") | (merged_df_temp["target_ac"]=="AA(Report.)") | (merged_df_temp["target_ac"]=="A(Report.)")),
                             "rgba(144,238,144,0.25)",
                             np.where(((merged_df_temp["source_ac"]=="BBB(Sanct.)") | (merged_df_temp["source_ac"]=="BB(Sanct.)") | (merged_df_temp["source_ac"]=="B(Sanct.)")|
                                      (merged_df_temp["target_ac"]=="BBB(Report.)") | (merged_df_temp["target_ac"]=="BB(Report.)") | (merged_df_temp["target_ac"]=="B(Report.)")),
                                      "rgba(255,215,0,0.20)",
                                      np.where(((merged_df_temp["source_ac"]=="CCC(Sanct.)") | (merged_df_temp["target_ac"]=="CCC(Report.)")),
                                      "rgba(240,128,128,0.25)","rgba(240,128,128,0.50)"))).tolist()
    
    merged_df_dict = merged_df.to_dict(orient="list")
    
    # Node Colors :
    color_map_key = {"AAA(Sanct.)" : "darkgreen", "AA(Sanct.)": "darkgreen", "A(Sanct.)" : "darkgreen",
                     "BBB(Sanct.)" : "gold", "BB(Sanct.)" : "gold", "B(Sanct.)" : "gold",
                     "CCC(Sanct.)" : "indianred", "D(Sanct.)" : "darkred",
                     "Commercial" : "midnightblue", "Corporate" : "midnightblue", "Retail" : "midnightblue", "SME": "midnightblue",
                     "AAA(Report.)" : "darkgreen", "AA(Report.)": "darkgreen", "A(Report.)" : "darkgreen",
                     "BBB(Report.)" : "gold", "BB(Report.)" : "gold", "B(Report.)" : "gold",
                     "CCC(Report.)" : "indianred", "D(Report.)" : "darkred"
                     }
    
    colors_list = pd.Series(unique_source_target).map(color_map_key).tolist()
    
    map_vals = pd.Series(unique_source_target)
    
    colors_list = map_vals.map(color_map_key)
    
    clean_labels = [g.removesuffix("(Sanct.)").removesuffix("(Report.)") for g in unique_source_target]
    
    #Sankey Plotting : 
    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node = dict(
            pad = 15,
            thickness = 20,
            line = dict(color="black", width = 0.5),
            label = clean_labels, #unique_source_target,
            x=[0.01, 0.01, 0.01, 0.01, 0.01, 0.01,0.01,           # 7 sanction rating AAA - CCC
               0.50, 0.50, 0.50, 0.50,                            # 4 segments 'Commercial', 'Corporate', 'Retail', 'SME'
               0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99],  # 8 reporting ratings AAA - D
            
            y=[0.01, 0.07, 0.18, 0.42, 0.7, 0.85, 0.95,          # 7 sanction rating AAA - CCC
               0.05, 0.20, 0.55, 0.9,                            # 4 segments 'Commercial', 'Corporate', 'Retail', 'SME'
               0.06, 0.12, 0.01, 0.4, 0.5, 0.25, 0.6,0.85],      # 8 reporting ratings AAA - D
            color = colors_list),
        link = dict(
            source = merged_df_dict["source"],
            target=merged_df_dict["target"],
            value = merged_df_dict["value"],
            color = linkcolorList))],
                   )
    
    fig.update_layout(
        title_text=f"Portfolio Credit rating Migration as on {reporting_date} for {col_name.removeprefix('Estimated_Rating_')} Scenario", 
        font_size=12,width=1150,height=790)

    # Uncomment to save the plot as an image :
    #fig.write_image("3_Rating_Distribution_Plot.png", width = 1000)
    #fig.write_html("3_Rating Distribution Plot.html")  # To save as an interactive html file
    
    fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)')
    
    fig.show()
    console.print(f" Total Customers as on {reporting_date} : {total}",style="Bold")



# ————————————————————————————————————————————————
# 4. Stage Migration Plot (Sankey Diagram) :
# ————————————————————————————————————————————————

def stage_migration(dataset = None, start_reporting_date = "2023-03-31",reporting_date="2023-06-30",scenario="Base"):

    if reporting_date < start_reporting_date:
        raise  ValueError("Start date should be before end date")
        
    segment_order = ["Commercial","Corporate", "Retail", "SME"]
    
    stage_order = [1,2,3] # Generates an index value of [1,2,3], representing the Stage.
    
    stage_dict = {s: "S"+str(s) for s,v in enumerate(stage_order,start=1)}
    
    stages = ["S"+str(s) for s in stage_order] # Creates a list : [S1,S2,S3]
    
    # The Source DataFrame :
    df1 = dataset[dataset["Reporting_Date"]==start_reporting_date]
    start_countCust = df1["Customer_ID"].nunique()
    df1 = df1.filter(regex=f"Customer_ID|Segment|Stage_{scenario}")
    df1 = df1.groupby([f"Stage_{scenario}","Segment"])["Customer_ID"].count().reset_index()
    df1.columns= ["source","target","value"]
    
    df1["source"] = df1["source"].map(stage_dict)
    df1["source"] = df1["source"] +"_B"
    
    # The Destination DataFrame :
    df2 = dataset[dataset["Reporting_Date"]==reporting_date]
    end_countCust = df2["Customer_ID"].nunique()
    df2 = df2.filter(regex=f"Customer_ID|Segment|Stage_{scenario}")
    df2 = df2.groupby(["Segment",f"Stage_{scenario}"])["Customer_ID"].count().reset_index()
    df2.columns = ["source","target","value"]
    df2["target"] = df2["target"].map(stage_dict)
    df2["target"] = df2["target"] +"_F"
    
    # Creating a New DataFrame Combining the 2 dataframe created above : 
    
    combinedf = pd.concat([df1,df2],axis=0)
    
    # Now, what we want is for all "Source" and "target" variables to have numbers, which is what plotly requires.
    # We want the unique values to be unique.
    
    unqiue_soure_target = list(pd.unique(combinedf[["source","target"]].values.ravel("k")))
    unqiue_soure_target
    
    mapping_dict  = {k:v for v, k in enumerate(unqiue_soure_target)}
    # Personal note -  Output : {'S1': 0, 'S2': 1, 'S3': 2,'Commercial': 3, 'Corporate': 4, 'Retail': 5, 'SME': 6}
    
    combinedf["source"] = combinedf["source"].map(mapping_dict)
    combinedf["target"] = combinedf["target"].map(mapping_dict)
    
    # Now, we have to modify the dataframe into a Dictionary where every column name is a Key and every value inside that column is a list. 
    combinedf_dict = combinedf.to_dict(orient="list")
    
    # The Alluvial Diagram / Sankey Diagram : 
    # Defining the labels :
    label = stages + segment_order+ stages

    stage_color_map = {
    "S1": "rgba(0, 100, 0, 0.4)",       # Dark Green with 40% transparency
    "S2": "rgba(255, 215, 0, 0.4)",     # Gold with 40% transparency
    "S3": "rgba(205, 92, 92, 0.4)"      # Indian Red with 40% transparency
    }
    
    colorlist = []
    colorlist = np.where((combinedf["source"] == 0) | (combinedf["target"]==7),
                                  "rgba(144, 238, 144, 0.25)",
                                  np.where((combinedf["source"] == 1) | (combinedf["target"]==8),
                                  "rgba(255, 215, 0, 0.25)","rgba(240, 128, 128, 0.25)")).tolist()
    
    clean_labels = [g.removesuffix("_B").removesuffix("_F") for g in unqiue_soure_target]
    
    fig = go.Figure(data = [go.Sankey(
        arrangement="snap",
        node = dict(
            thickness=20,line = dict(color="black", width=0.5),
            label= clean_labels, # unqiue_soure_target,
            color=["darkgreen","gold","indianred","midnightblue","midnightblue","midnightblue","midnightblue","darkgreen","gold","indianred"]
            ),
        link = dict(
            source= combinedf_dict["source"], # The Dictionary we created from out DataFrame
            target=combinedf_dict["target"],
            value = combinedf_dict["value"],
            color=colorlist
        ))])
    fig.update_layout(
            title_text=f"Portfolio Stage Migration between {start_reporting_date} and {reporting_date} under {scenario} Scenario", 
            font_size=12,width=1150,height=790)
    
    # Uncomment to save the plot as an image :
    #fig.write_image("4_Stage Migration.png", width = 1000)
    
    fig.show()
    
    console.print(f"Total Customers on {start_reporting_date} : {start_countCust}\n"
                  f"Total Customers on {reporting_date} : {end_countCust}")

# ————————————————————————————————————————————————
# 5. Stage migration of borrowers within a Sector :
# ————————————————————————————————————————————————

def stage_sector_migration(dataset = None, start_reporting_date = "2023-03-31",reporting_date="2023-06-30",scenario="Base", segment="Retail"):

    if reporting_date < start_reporting_date:
        raise  ValueError("Start date should be before end date")
        
    sector_order = dataset[dataset["Segment"]==segment]["Sector"].unique().tolist()
    stage_order = [1,2,3]
    stage_dict = {s: "S"+str(s) for s,v in enumerate(stage_order,start=1)}
    
    stages = ["S"+str(s) for s in stage_order] # Creates a list : [S1,S2,S3]
    
    # The Source DataFrame :
    df1 = dataset[(dataset["Reporting_Date"]==start_reporting_date) & (dataset["Segment"]==segment)]
    start_countCust = df1["Customer_ID"].nunique()
    df1 = df1.filter(regex=f"Customer_ID|Sector|Stage_{scenario}")
    df1 = df1.groupby([f"Stage_{scenario}","Sector"])["Customer_ID"].count().reset_index()
    df1.columns= ["source","target","value"]
    
    df1["source"] = df1["source"].map(stage_dict)
    df1["source"] = df1["source"] +"_B"
    
    
    # The Destination DataFrame :
    df2 = dataset[(dataset["Reporting_Date"]==reporting_date) & (dataset["Segment"]==segment)]
    end_countCust = df2["Customer_ID"].nunique()
    df2 = df2.filter(regex=f"Customer_ID|Sector|Stage_{scenario}")
    df2 = df2.groupby(["Sector",f"Stage_{scenario}"])["Customer_ID"].count().reset_index()
    df2.columns = ["source","target","value"]
    df2["target"] = df2["target"].map(stage_dict)
    df2["target"] = df2["target"] +"_F"
     
    # # Creating a New DataFrame Combining the 2 dataframe created above : 
    
    combinedf = pd.concat([df1,df2],axis=0)
    
    # # Now, what we want is for all "Source" and "target" variables to have numbers, which is what plotly requires.
    # # We want the unique values to be unique.
    
    unqiue_source_target = list(pd.unique(combinedf[["source","target"]].values.ravel("k")))
    
    mapping_dict  = {k:v for v, k in enumerate(unqiue_source_target)}
    # # Personal note -  Output : {'S1_B': 0,'S2_B': 1, 'S3_B': 2, 'Logistics': 3, 'Real Estate': 4, 'Retail Trade': 5, 'Wholesale': 6, 'S1_F': 7, 'S2_F': 8, 'S3_F': 9}
    
    combinedf["source"] = combinedf["source"].map(mapping_dict)
    combinedf["target"] = combinedf["target"].map(mapping_dict)
    
    # # Now, we have to modify the dataframe into a Dictionary where every column name is a Key and every value inside that column is a list. 
    combinedf_dict = combinedf.to_dict(orient="list")
    
    # # The Alluvial Diagram / Sankey Diagram : 
    # # Defining the labels :
    label = stages + sector_order + stages

    # used to create a list for links color: 
    reverse_mapping_dict = {k:v for k,v in enumerate(unqiue_source_target)}
    #print(f" Reverse reverse_mapping_dict : {reverse_mapping_dict}")
    
    combinedf_temp = combinedf.copy()
    combinedf_temp["source_ac"] = combinedf_temp["source"].map(reverse_mapping_dict)
    combinedf_temp["target_ac"] = combinedf_temp["target"].map(reverse_mapping_dict)

    colorlist = []
    colorlist = np.where((combinedf_temp["source_ac"] == "S1_B") | (combinedf_temp["target_ac"]=="S1_F"),
                                  "rgba(144, 238, 144, 0.25)",
                                  np.where((combinedf_temp["source_ac"] == "S2_B") | (combinedf_temp["target_ac"]=="S2_F"),
                                  "rgba(255, 215, 0, 0.25)","rgba(240, 128, 128, 0.25)")).tolist()
    
    clean_labels = [g.removesuffix("_B").removesuffix("_F") for g in unqiue_source_target]

    node_colorList = ["darkgreen","gold","indianred"]+ ["midnightblue"]*len(sector_order)+["darkgreen","gold","indianred"]
    
    fig = go.Figure(data = [go.Sankey(
        arrangement="snap",
        node = dict(
            thickness=20,line = dict(color="black", width=0.5),
            label= clean_labels, # unqiue_soure_target,
            color=node_colorList
            ),
        link = dict(
            source= combinedf_dict["source"], # The Dictionary we created from out DataFrame
            target=combinedf_dict["target"],
            value = combinedf_dict["value"],
            color=colorlist
        ))])
    fig.update_layout(
            title_text=f"Portfolio Stage Migration within the {segment} Segment between {start_reporting_date} and {reporting_date} for {scenario} Scenario", 
            font_size=12,width=1150,height=790)

    # Uncomment to save the plot as an image :
    # fig.write_image("5_Stage Sector migration.png", width= 1200)
    
    fig.show()
    
    console.print(f"Total Customers on {start_reporting_date} : {start_countCust}\n"
                  f"Total Customers on {reporting_date} : {end_countCust}")

# ————————————————————————————————————————————————
# 6. Rating Heatmap : 
# ————————————————————————————————————————————————

def rating_heatmap(dataset = None, reporting_date = "2023-03-31",scenario="Base"):
    
    scen = "Estimated_Rating_"+scenario
    
    seg_data = dataset[dataset["Reporting_Date"]==reporting_date]
    
    df3 = (seg_data.groupby(["Sector", scen])["Customer_ID"].count().reset_index().rename(columns={"Customer_ID": "Count"}))
    
    heatmap_data = df3.pivot(index="Sector", columns=scen, values="Count").fillna(0)
    
    rating_order = ["AAA","AA","A","BBB","BB","B","CCC","D"]
    heatmap_data = heatmap_data.reindex(columns=rating_order)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    sns.heatmap(heatmap_data,annot=True,fmt=".0f",cmap="Blues",linewidths=0.5,ax=ax)
    
    ax.set_title(f"Loan Count by Sector and Reported Rating : {reporting_date} for {scenario} Scenario", fontsize=12,
                fontweight='semibold', color='#475569')
    
    ax.set_xlabel("Estimated Rating", fontsize=10,fontweight='bold', color='#475569')
    ax.set_ylabel("Sector")
    plt.tight_layout()
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")

    # Uncomment to save the plot as an image :
    # plt.savefig("6_Rating heatmap.png", bbox_inches="tight",dpi=400)
    
    plt.show()

# ————————————————————————————————————————————————
# 7. Sector - Segment Rating Quality Comparison
# ————————————————————————————————————————————————

def plot_sector_segment_quality_stacked(dataset, reporting_date="2023-03-31", check_type = "Sector", scenario = "Base"):
    # 1. Filter dataset for reporting date
    sub_df = dataset[dataset["Reporting_Date"] == reporting_date]

    # 2. Pivot data: Sector vs Rating
    ct = pd.crosstab(sub_df[check_type], sub_df[f"Estimated_Rating_{scenario}"]).reindex(columns=rating_order, fill_value=0)

    # 3. Normalize row-wise (Sector totals sum to 100%)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

    # 4. Sort sectors by proportion of High-Risk / Non-Investment Grade (e.g., CCC or B & below)
    # This automatically pushes the worst-performing sectors to the top of the chart
    high_risk_cols = [r for r in rating_order if r in ["B", "CCC", "CC", "C", "D"]]
    
    if high_risk_cols:
        ct_pct["_sort_key"] = ct_pct[high_risk_cols].sum(axis=1)
        ct_pct = ct_pct.sort_values("_sort_key", ascending=True).drop(columns=["_sort_key"])

    rating_colors = sns.color_palette("Spectral_r", n_colors=len(rating_order))

    # 6. Plot stacked bars
    fig, ax = plt.subplots(figsize=(10, 6))
    ct_pct.plot(kind="barh", stacked=True, color=rating_colors, ax=ax, width=0.75)

    # 7. Formatting & Annotations
    ax.set_title(f"{check_type} Quality Profile (% Credit Rating Composition)\nAs of {reporting_date} for {scenario} Scenario",
                 fontsize=12, fontweight="bold", pad=15,loc="left", color='#475569')
    
    ax.set_xlabel(f"Proportion of {check_type} (%)", fontsize=10,fontweight='semibold', color='#475569')
    ax.set_ylabel("")
    ax.set_xlim(0, 100)

    # Clean legend positioning outside plot
    ax.legend(title="Rating Grade",bbox_to_anchor=(1.02, 1),loc="upper left",frameon=False)

    sns.despine(top=True, right=True)
    plt.tight_layout()

    # Uncomment to save the plot as an image :
    
    # if check_type =="Sector":
    #     plt.savefig("7.1_Sector_Quality_Stacked.png", bbox_inches="tight", dpi=400)
    # else:
    #     plt.savefig("7_Segment_Quality_Stacked.png", bbox_inches="tight", dpi=400)
    
    plt.show()


# ————————————————————
# Displaying all Plots : 
# ————————————————————

# 1. 

console.print(f"==================================================================================\n"
              f"\nPlot 1 : Scenario ECL\n"
              f"==================================================================================\n"
              ,style="bold")

display(widgets.interactive(ecl_reporting_date,
                            dataset = widgets.fixed(final_active_book),  # The dataset 
                            reporting_date=widgets.Select(
                                options=reporting_dates_list,
                                description="Reporting Date",
                                style={'description_width': '120px'}
                            )))

# 2. 

console.print(f"==================================================================================\n"
              f"\nPlot 2 : Credit rating Distribution Plot by Reporting Date\n"
              f"==================================================================================\n"
              ,style="bold")

display(widgets.interactive(rating_dist_rep_date,
                            dataset = widgets.fixed(final_active_book),     # The dataset 
                            reporting_date=widgets.Select(
                                value=reporting_dates_list[2],
                                options=reporting_dates_list, description="Reporting Date", style={"description_width":"120px"})))

# 3.

console.print(f"==================================================================================\n"
              f"\nPlot 3 : Credit rating migration from Origination Date\n"
              f"==================================================================================\n"
              ,style="bold")

display(widgets.interactive(
    credit_migration_diagram,
    dataset = widgets.fixed(final_active_book), # The dataset 
    reporting_date=widgets.Select(
        options=reporting_dates_list,
        description='Reporting Date',
        style={'description_width': 'initial'}
    ),
    scenario = widgets.ToggleButtons(
        options=scen,
        description="Scenario",
        button_style="info"
    )))

# 4.

console.print(f"==================================================================================\n"
              f"\nPlot 4 : Stage migration of loans within each segment between two date range\n"
              f"==================================================================================\n"
              ,style="bold")

start_reporting_date_widget = widgets.Select(options=reporting_dates_list,description="Start Date", style={"description_width":"initial"})
reporting_date_widget = widgets.Select(options=reporting_dates_list,value=reporting_dates_list[1],description="End Date", style={"description_width":"initial"})
scenario_widget = widgets.ToggleButtons(options=scen,description="Scenario",button_style="Info")


ui_output = widgets.interactive_output(stage_migration,
                                       {
        'dataset': widgets.fixed(final_active_book), # The dataset 
        'start_reporting_date': start_reporting_date_widget,
        'reporting_date': reporting_date_widget,
        'scenario': scenario_widget
    }
)

dates_hbox = widgets.HBox([start_reporting_date_widget,reporting_date_widget])
final_layout = widgets.VBox([dates_hbox, scenario_widget,ui_output])

display(final_layout)

# 5.

console.print(f"==================================================================================\n"
              f"\nPlot 5 : Stage migration of loans within a sector between two date range\n"
              f"==================================================================================\n"
              ,style="bold")

segments = final_active_book["Segment"].unique().tolist()

start_reporting_date_widget = widgets.Select(options=reporting_dates_list,description="Start Date", style={"description_width":"initial"})
reporting_date_widget = widgets.Select(options=reporting_dates_list,value=reporting_dates_list[1],description="End Date", style={"description_width":"initial"})
scenario_widget = widgets.ToggleButtons(options=scen,description="Scenario",button_style="Info")
Segment_widget = widgets.ToggleButtons(options=segments,description="Segment",button_style="Info")

ui_interact = widgets.interactive_output(stage_sector_migration,
                                         {"dataset" : widgets.fixed(final_active_book),  # The dataset 
                                          "start_reporting_date" : start_reporting_date_widget,
                                          "reporting_date" : reporting_date_widget,
                                          "scenario" : scenario_widget,
                                          "segment":Segment_widget})

dates_hbox = widgets.HBox([start_reporting_date_widget,reporting_date_widget])
final_layout = widgets.VBox([dates_hbox, scenario_widget,Segment_widget, ui_interact])

display(final_layout)

# 6. 

console.print(f"\n================================================================================== \n"
              f"\nPlot 6 : Number of borrowers in each rating grouped by sector\n"
              f"\n================================================================================== \n",style="bold")

display(widgets.interactive(rating_heatmap,
                            dataset = widgets.fixed(final_active_book),   # The dataset 
                            reporting_date = widgets.Select(options=reporting_dates_list, description="Reporting Date",style={"description_width":"initial"}),
                            scenario=widgets.ToggleButtons(options=scen, description="Scenario",button_style="info",style={"description_width":"initial"})
                           ))

# 7. 

console.print(f"\n================================================================================== \n"
              f"\nPlot 7 : Sector - Segment Rating Quality Comparison\n"
              f"\n================================================================================== \n",style="bold")

types = ["Segment","Sector"]

display(widgets.interactive(plot_sector_segment_quality_stacked,
                            dataset = widgets.fixed(final_active_book),  # The dataset 
                            reporting_date = widgets.Select(options = reporting_dates_list, value= reporting_dates_list[1],style={"description_width":"initial"}),
                            check_type = widgets.ToggleButtons(options = types, button_style="info",style={"description_width":"initial"},description="  "),
                            scenario = widgets.ToggleButtons(options = scen, button_style="info", style={"description_width":"initial"}, description="Scenario")
                           ))

In [ ]:
"""
Data Validation Checks on the Model : 
"""
check_dict = {}
metrics = ["CCF", "ECL", "PD", "LGD", "EAD"]
required_probation_period = 2
passed = "[bold green] Passed [/bold green]"
notpass = "[bold red] Failed [/bold red]"
datalen = len(forecasting_data)
check = 0

def validate_data(dataset = None):

    if data is None:
        raise ValueError("No data provided")
    for idx, s in enumerate(scen,start=1):
    
        """
        Check 1 : Identifying loans that are marked as Stage 2 accounts and DPD bucket is 30-59 DPD
        """
        check = 1
        key_val = f"Check_{check}_{s}"
        
        checking_data = dataset[(dataset[f"DPD Bucket_{s}"].isin(["30-59 DPD","60-89 DPD"])) & (dataset[f"Estimated_Rating_{s}"]!="D") & 
                    (dataset[f"Stage_{s}"]==1)][["Customer_ID","Origination_Rating",f"DPD Bucket_{s}",f"Stage_{s}",f"Estimated_Rating_{s}"]]
        
        Datalength = len(checking_data)
        
        check_dict[key_val] = f" Number of observations that are 30-59 DPD or 60-89 DPD but are still in Stage 1 : {Datalength} ; Unique loan count : {checking_data["Customer_ID"].nunique()}"
        
        """
        Check 2 : All loans that are 90+ DPD are marked as "D".
        """
        check += 1
        key_val = f"Check_{check}_{s}"
    
        check_dict[key_val] = f"Number of loans that are 90+ DPD but rating is not marked as 'D' : {dataset[(dataset[f"DPD Bucket_{s}"]=='90+ DPD') & (final_active_book[f"Estimated_Rating_{s}"]!='D')].count().iloc[0]}\n"
        
        """
        Check 3 : Loans that were originated upto 3 months before the first reporting date should not be in the delinquency buckets.
        """
    
        check += 1
        key_val = f"Check_{check}_{s}"
    
        val = dataset[(dataset["Quarter_Index"]==0) & (dataset["Q_Count_for_Rating_Est."]==0) & 
              ((dataset[f"DPD Bucket_{s}"]=="90+ DPD") | (dataset[f"DPD Bucket_{s}"]=="30-59 DPD") |
               (dataset[f"DPD Bucket_{s}"]=="60-89 DPD"))].count().iloc[0]
        
        check_dict[key_val] = f" Newly originated loans right before the first reporting date that are in delinquent buckets: {val}"
    
        """
        Check 4 : DPD Buckets do not have any other value apart from "Current", "30-59 DPD", "60-89 DPD" or "90+ DPD":
        """
    
        check += 1
        key_val = f"Check_{check}_{s}"
        check_dict[key_val] = f" Does the DPD Bucket for {s} scenario only have values from the list ['Current', '30-59 DPD', '60-89 DPD', '90+ DPD'] : {"[bold green]Yes[/bold green]" if final_active_book[f"DPD Bucket_{s}"].isin(['Current','30-59 DPD','60-89 DPD','90+ DPD']).all()==True else "[bold red]No[/bold red]"}\n Unique values in the data : {dataset[f"DPD Bucket_{s}"].unique()}"
    
        """
        Check 5 : Stage do not have any other value apart from 1, 2 or 3:
        """
    
        check += 1
        key_val = f"Check_{check}_{s}"
        check_dict[key_val] = f" Does the Stage for {s} scenario only have values of 1, 2, and 3 : {"[bold green]Yes[/bold green]" if dataset[f"Stage_{s}"].isin([1,2,3]).all()==True else "[bold red]No[/bold red]"}\n Unique values in the data : {dataset[f"Stage_{s}"].unique()}"
    
        """
        Check 6 :  Checking if Macroeconomic Forecast values are correctly mapped
        """
        check +=1
        key_val = f"Check_{check}_{s}"
        
        check_dict[key_val] = "\n".join([f" Is the value of macroeconomic forecast correctly mapped? : {"[bold green]Yes[/bold green]" if np.array_equal(np.array(mev_forecast[mev_forecast['observation_date']>='2023-03-31'][f'Z_{s}']),np.array(final_active_book[f'Z_{s}'].drop_duplicates())) else "[bold red]No[/bold red]"}"])
    
        
    """
    Check 7  : Only revolving loans should have a total of 12 rows if maturity date if before the end of the reporting period. Note no new loans are originated 
              during the observation period thus, no need to check for sanction date.
    """
    
    check +=1
    key_val = f"Check_{check}"
    check_dict[key_val] = f" Expected loan category should be 'Revolving' {passed if dataset[(dataset['Total_Periods']==12) & (dataset['Maturity_Date']<'2025-12-31')]['Type_of_Loan'].unique()=="Revolving" else notpass}"
    
    """
    Check 8 : Checking the probation period is correctly applied :
    """
    
    probation_check = required_probation_period  # = 2
    
    
    # New variables : 
    value_counts = {}
    for s in scen :
        subset_data = dataset[dataset[f"Estimated_Rating_{s}"].notna()].reset_index(drop=True)
        
        incorrect_location = []
        correct_count = 0
        incorrect_count = 0
        i = 0
        while i < len(subset_data):
            row = subset_data.loc[i]
            trigger  = (row[f"DPD Bucket_{s}"] == "90+ DPD") and \
                            (row[f"Estimated_Rating_{s}"]  == "D")
            
            if trigger:
                # Check next probation_check rows
                for j in range(1, probation_check + 1):
                    next_i = i + j
                    
                    # Guard: don't go out of bounds or into next customer
                    if next_i >= len(subset_data):
                        break
                    if subset_data.loc[next_i, "Quarter_Index"] == 0:
                        break  # new customer starts — stop probation check
                    
                    next_row = subset_data.loc[next_i]
                    
                    # Base
                    if trigger:
                        if next_row[f"Estimated_Rating_{s}"] == "D":
                            correct_count += 1
                        else:
                            incorrect_count += 1
                            incorrect_location.append(i)            
                i += probation_check  # skip ahead past probation window
    
            else:
                i += 1
        value_counts[s] = {}        
        value_counts[s]["Correct Count"] = correct_count
        value_counts[s]["Incorrect Count"] = incorrect_count
        value_counts[s]["Incorrect_Locations"] =incorrect_location 
        incorrect_location = []    
    # ——————————————————————
    # Printing the results for checks 1 to 7 : 
    # ——————————————————————
    
    for key, val in check_dict.items():
        console.print(f"==============================\n",
                      f"{key} Scenario\n",
                      f"==============================\n",
                      f"{val}")


    # ——————————————————————
    # Printing the results for check 8 : 
    # ——————————————————————
    for key in value_counts.keys():
        console.print(f"Correct Count : {value_counts[key]["Correct Count"]}\n",
                      f"Incorrect Count : {value_counts[key]["Incorrect Count"]}\n",
                      f" Incorrect list : {value_counts[key]["Incorrect_Locations"]}")
    
    """
    Check 9 : Monte Carlo Simulation : Checking whether the active and charge off books observations sum to the input data observations : 
    """
    correct_count = 0
    correct_item_list = []
    incorrect_count = 0
    incorrect_item_list = []
    
    total_set = len(books_MC.keys()) * len(scen)
    
    for key, value in books_MC.items():
        for s in scen:
            active_book_obs = books_MC[key][s]["active"].shape[0]
            charge_off_book_obs = books_MC[key][s]["charge_off"].shape[0]
    
            total = active_book_obs +charge_off_book_obs
            if total == final_active_book.shape[0] :
                correct_count+=1
                item = f"{key}_Scenario : {s}"
                correct_item_list.append(item)
            else:
                incorrect_count+=1
                item = f"{key} Scenario : {s}"
                incorrect_item_list.append(item)

    console.print(f"============================\n"
                  f" Check 9 :\n"
                  f"============================")
    
    if correct_count != total_set:
        console.print(f" There is some inconsistency found in the total number of observations in the active and charge off book.\n Incorrect count : {incorrect_count}\n"
                     f" Where error was identified : {incorrect_item_list}", style="red bold")
    else:
        console.print(f" Upon merging the two books, the total number of observations correctly match the number of observations in the input data to the ECL_sim function",style="green bold")

# ——————————————————————————————————————
# Call the validate_data function : 
# ——————————————————————————————————————
validate_data(dataset = final_active_book)

# Interest Rate Modelling

In [ ]:
temp = forecasting_data.copy()
forecasting_data["Sanction_Quarter_Group"] = forecasting_data["Sanction_Date"].dt.to_period("Q")

fed_funds_rate = pd.read_excel("Sample_Portfolio.xlsm", sheet_name = "FEDFUNDS")

average_vals = pd.DataFrame(forecasting_data.groupby(["Sanction_Quarter_Group","Type_of_Loan"])["Origination_Interest_Rate"].mean()).reset_index()

average_vals_amortising = average_vals[average_vals["Type_of_Loan"]=="Amortising"].reset_index(drop=True)
gap_amortising = average_vals_amortising["Origination_Interest_Rate"] - fed_funds_rate["FEDFUNDS"]
avg_gap_amortising = gap_amortising.mean()

average_vals_bullet = average_vals[average_vals["Type_of_Loan"]=="Bullet"].reset_index(drop=True)
gap_bullet = average_vals_bullet["Origination_Interest_Rate"] - fed_funds_rate["FEDFUNDS"]
avg_gap_bullet = gap_bullet.mean()

average_vals_revolving = average_vals[average_vals["Type_of_Loan"]=="Revolving"].reset_index(drop=True)
gap_revolving = average_vals_revolving["Origination_Interest_Rate"] - fed_funds_rate["FEDFUNDS"]
avg_gap_revolving = gap_revolving.mean()

fig, ax = plt.subplots(nrows=2,ncols=2,figsize=(12,6))

ax[0,0].plot(fed_funds_rate["FEDFUNDS"], label="Federal Funds Rate", color="darkblue")
ax[0,0].plot(average_vals_amortising["Sanction_Quarter_Group"].astype(str), average_vals_amortising["Origination_Interest_Rate"],
             label="Avg. Amortising I/Y", color="Khaki")

ax[0,0].plot(gap_amortising, color="red", linestyle="--", label="Interest charged above fed funds")
ax[0,0].axhline(y = avg_gap_amortising, linestyle="--", color="black", lw=1)

ax[0,0].set_title("Amortising Loan Interest Rate v/s Fed Funds Rate")

ax[0,1].plot(average_vals_bullet["Sanction_Quarter_Group"].astype(str), average_vals_bullet["Origination_Interest_Rate"],
             label="Avg. Bullet I/Y", color="Indianred")

ax[0,1].plot(fed_funds_rate["FEDFUNDS"],color="darkblue")
ax[0,1].plot(gap_bullet, color="red", linestyle="--")
ax[0,1].axhline(y = avg_gap_bullet, linestyle="--", color="black", lw=1)
ax[0,1].set_title("Bullet Loan Interest Rate v/s Fed Funds Rate")

ax[1,0].plot(average_vals_revolving["Sanction_Quarter_Group"].astype(str), average_vals_revolving["Origination_Interest_Rate"],
             label="Avg. Revolving I/Y", color="green")

ax[1,0].plot(fed_funds_rate["FEDFUNDS"], color="darkblue")
ax[1,0].plot(gap_revolving, color="red", linestyle="--")
ax[1,0].axhline(y = avg_gap_revolving, linestyle="--", color="black", lw=1, label="Average spread")
ax[1,0].set_title("Revolving loan Interest rate v/s Fed Funds Rate")

ax[1,1].set_visible(False)


# Setting the x axis values (showing years and not quarters)
quarters = average_vals_amortising["Sanction_Quarter_Group"].astype(str).tolist()
years = [q.split("Q")[0] for q in quarters]
unique_years = sorted(list(set(years)))
tick_indices = [years.index(y) for y in unique_years]

for current_ax in [ax[0,0], ax[0,1], ax[1,0]]:
    current_ax.set_xticks(tick_indices)
    current_ax.set_xticklabels(unique_years, fontsize=10,rotation=45)
    current_ax.axvspan(tick_indices[10],tick_indices[11],color="grey",alpha=0.15)
    
fig.legend(loc="center", bbox_to_anchor=(0.65,0.3), fontsize=11)
plt.tight_layout()
plt.show()

avg_amortising_int = average_vals_amortising["Origination_Interest_Rate"].mean()
avg_bullet_int = average_vals_bullet["Origination_Interest_Rate"].mean()
avg_revolving_int = average_vals_revolving["Origination_Interest_Rate"].mean()

print(f" avg_amortising_int : {avg_amortising_int*100:.4f}%\n"
      f" avg_bullet_int : {avg_bullet_int*100:.4f}%\n"
      f" avg_revolving_int : {avg_revolving_int*100:.4f}%\n")

print(f" Average Interest Rate Charged Above the Fed Funds for Amortising Loan : {avg_gap_amortising*100:.4f}%\n"
      f" Average Interest Rate Charged Above the Fed Funds for Bullet Loan : {avg_gap_bullet*100:.4f}%\n"
      f" Average Interest Rate Charged Above the Fed Funds for Revolving Loan : {avg_gap_revolving*100:.4f}%")
